# WaveSSM-TriNet — revision 2 (Reviewer 2)

Run the next cell **as one cell** (it launches one worker process per GPU from its own source). Settings: Accelerator = GPU T4 x2, Internet ON.

| Goal | Environment variables before the cell |
|---|---|
| Everything for the headline results (full + softmax head, 5 seeds, CV) | `R2_STAGE=heads` |
| Table-11 ablation over seeds (7 variants x 3 seeds) | `R2_STAGE=ablation` |
| Alternative / parameter-matched fusion + capacity baselines | `R2_STAGE=fusion` |
| Evidence-ceiling probe (clamp 100) | `R2_STAGE=clamp` |
| Continue after a Kaggle session ended | `R2_RESUME_FROM=/kaggle/input/<previous-output>` |
| Re-make every table/figure on CPU from saved predictions | `R2_EVAL_ONLY=1` |
| 2-minute synthetic smoke test | `RUN_MODE=smoke` |

In a notebook set them with `import os; os.environ["R2_STAGE"] = "heads"` in a **separate cell before** this one. See `README_R2.md`.


In [1]:
"""
WaveSSM-TriNet (revised) -- Kaggle T4 pipeline, BRISC 2025 (+ Nickparvar external)
================================================================================
Changes versus the first submission, keyed to Reviewer-1 comment numbers:
  #12/#13  TCAFG now attends over TOKEN SEQUENCES (49 tokens/branch), six directional
           multi-head cross-attentions, a gate per pair, attention-entropy logging.
  #20      Capacity/fusion baselines: concat-MLP, standard MHA (transformer) fusion, gated
           fusion, EfficientNet-only (B3 / B4); branch-subset ablations. See VARIANTS.
  #21      Multi-seed runs, mean/SD/95% CI, paired tests (EVALUATION UTILITIES section).
  #3/#4/#7 Near-duplicate detection, duplicate clusters used as groups, group-aware
           stratified train/val split, group-aware CV. Official BRISC test split is
           NEVER used for checkpoint selection (it is scored once).
  #5/#6    Every table/figure is generated from saved prediction CSVs (EVALUATION UTILITIES section).
  #15      Evidential loss: float32, logit clamp, max-evidence cap, hard labels in the KL
           term, configurable label smoothing and KL annealing. Loss is the expected-MSE
           (Brier-type) form -- the paper text must say so, not "cross-entropy".
  #16-#18  ECE/Brier/NLL, reliability, temperature scaling on VAL, error-detection AUROC,
           risk-coverage, high-confidence-error table.
  #22      Branch occlusion (mean-token substitution) + exact 3-player Shapley values.
  #23      Grad-CAM for ALL three branches w.r.t. the full-model decision (+ optional
           mask hit-rate if BRISC masks are available).
  #24      Stratified t-SNE (+UMAP if installed).
  #25/#28  Intensity/contrast shift statistics, external per-class uncertainty.
  #26      FlopCounterMode (counts F.conv2d and attention matmuls) + analytic scan FLOPs +
           measured latency / throughput / peak memory.
  #27      Exact protocol dumped to run_config.json; LR trajectory saved; pretrained
           weights REQUIRED (no silent fallback to random init / other backbones).
  #8       Haar DWT unit-tested against PyWavelets.
  #29      Split manifests, predictions, configs, environment saved to OUT_DIR.

SINGLE-FILE, KAGGLE GPU T4 x2 VERSION (eval_utils.py merged in).  Paste into ONE notebook cell and run.
  * Two worker processes, one per T4 (CUDA_VISIBLE_DEVICES), pull (variant, seed) jobs from a shared queue ->
    ~2x throughput with the SAME protocol (batch size, seeds, epochs) as a single-GPU run. No DataParallel.
  * A wall-clock budget (CFG["time_budget_h"], default 11.5 h) decides which jobs still fit; jobs are ordered by priority,
    so what is dropped (if anything) is the least important. Finished runs are never retrained (resumable).
  * RUN_MODE: "paper" (default: 5 seeds x 50 ep + 5-fold CV) | "quick" (3 seeds x 30 ep) | "smoke" (2-min synthetic test).
  * Do not put %%magics or other cells' code in the same cell; keep Internet ON (timm weights) and Accelerator = GPU T4 x2.

Revision 2 -- Reviewer-2 comments R2-1 ... R2-11 (implemented below and in the REVIEWER-2 EXTENSION BLOCK):
  R2-1/9   one reporting vocabulary (single run | mean +- SD over n seeds | CV), registry of canonical numbers (r2_registry.json), GFLOPs incl. scan,
           one branch naming (WSDB/SSSB/ESSB) in every figure.
  R2-2     every ablation variant over >= 3 seeds (CFG ablation_seeds); full + softmax_head over 5 seeds; paired differences, 95% t-intervals,
           Cohen's d_z, Holm, 'supported' flag, forest plot.
  R2-3     Wilson / t / cluster-bootstrap intervals, exact McNemar, Wilcoxon + its minimum attainable p, Shapiro note, unit-of-analysis rules.
  R2-4     external class-wise table, specificity, balanced accuracy, prevalence-reweighted metrics, error patterns.
  R2-5     calibration (raw + TS), tie-aware selective prediction, evidence-saturation diagnostics, relaxed-clamp probe (variant full_clamp100),
           reliability grid, class-wise uncertainty distributions.
  R2-6     split manifest (+ CV folds), per-run configs, SHA-256 checksums, REPRODUCE.md, resume across Kaggle sessions (R2_RESUME_FROM),
           analysis-only mode (R2_EVAL_ONLY=1).
  R2-7     similarity-stratified accuracy (max cosine similarity of each test image to the training set).
  R2-8     failure-case Grad-CAM, Grad-CAM mass outside the head / in corners, pointing game (mask_root), accuracy by plane/sequence,
           t-SNE by plane, linear probes, test-time perturbations.
  Staging on Kaggle:  R2_STAGE = heads | ablation | fusion | clamp | all   (finished jobs on disk are skipped)
"""
# WAVESSM-TRINET-SCRIPT
import os, sys, json, math, time, copy, random, warnings, subprocess, traceback
from itertools import combinations
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.checkpoint import checkpoint
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import transforms
from PIL import Image
import timm
from sklearn.model_selection import StratifiedGroupKFold
from scipy import stats
from sklearn.metrics import (confusion_matrix, roc_auc_score, roc_curve, auc,
                             cohen_kappa_score, matthews_corrcoef,
                             precision_recall_fscore_support, average_precision_score)
from sklearn.preprocessing import label_binarize
import seaborn as sns


warnings.filterwarnings("ignore")
SMOKE = os.environ.get("SMOKE", "0") == "1" or os.environ.get("RUN_MODE", "") == "smoke"


# ══════════════════════════════════════════════════════════════════════════════
#  REVIEWER-2 EXTENSION BLOCK  (torch-free; shipped inside the training script AND as r2_analysis.py)
#  Everything below works from the saved prediction CSVs + split_manifest.csv, so every number in
#  the revised tables can be regenerated on a CPU without retraining.
#
#   R2-1/R2-9  one reporting convention (r2_protocol), canonical-number registry, consistency checks
#   R2-2       multi-seed ablation table: paired differences, 95% t-intervals, Cohen's d_z, Holm, "supported" flag
#   R2-3       Wilson / t-interval / cluster-bootstrap CIs, exact McNemar, Wilcoxon (with its minimum p), Shapiro note
#   R2-4       external class-wise table, specificity, balanced accuracy, prevalence-reweighted metrics, error patterns
#   R2-5       calibration (raw + temperature-scaled), tie-aware selective prediction, evidence-saturation diagnostics,
#              reliability grid, class-wise uncertainty distributions
#   R2-7       similarity-stratified accuracy (leakage sensitivity), de-duplicated test results
#   R2-8       accuracy by imaging plane / sequence, head-mask + Grad-CAM statistics, perturbation helpers, linear probe
# ══════════════════════════════════════════════════════════════════════════════
import os, re, json, math, glob, hashlib
import numpy as np
import pandas as pd
from scipy import stats as _st
import matplotlib.pyplot as plt
from sklearn.metrics import (roc_auc_score, precision_recall_fscore_support, matthews_corrcoef,
                             cohen_kappa_score, confusion_matrix, average_precision_score)
from sklearn.preprocessing import label_binarize

R2_CLASSES = ["glioma", "meningioma", "notumor", "pituitary"]
R2_BRANCH_LABEL = {"wt": "WSDB", "ssm": "SSSB", "cnn": "ESSB"}        # one name per branch in every figure/table
R2_ALPHA = 0.05


# ── 1. statistical primitives ────────────────────────────────────────────────
def r2_wilson(k, n, z=1.96):
    if n <= 0:
        return (float("nan"), float("nan"))
    p = k / n; d = 1 + z * z / n; c = p + z * z / (2 * n)
    a = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return ((c - a) / d, (c + a) / d)


def r2_t_interval(x, level=0.95):
    """Mean, SD and t-interval (n-1 d.f.) of a small sample (seeds)."""
    x = np.asarray(x, float); x = x[np.isfinite(x)]; n = len(x)
    if n == 0:
        return dict(n=0, mean=float("nan"), sd=float("nan"), lo=float("nan"), hi=float("nan"))
    m = float(x.mean())
    if n == 1:
        return dict(n=1, mean=m, sd=float("nan"), lo=float("nan"), hi=float("nan"))
    sd = float(x.std(ddof=1)); h = float(_st.t.ppf(0.5 + level / 2, n - 1) * sd / math.sqrt(n))
    return dict(n=n, mean=m, sd=sd, lo=m - h, hi=m + h)


def r2_dz(d):
    """Cohen's d_z for paired differences = mean(d) / SD(d)."""
    d = np.asarray(d, float); d = d[np.isfinite(d)]
    if len(d) < 2 or d.std(ddof=1) == 0:
        return float("nan")
    return float(d.mean() / d.std(ddof=1))


def r2_holm(p):
    """Holm step-down adjusted p-values (NaN-safe, order preserved)."""
    p = np.asarray(p, float); out = np.full(len(p), np.nan); idx = np.where(np.isfinite(p))[0]
    order = idx[np.argsort(p[idx])]; m = len(order); run = 0.0
    for rank, i in enumerate(order):
        run = max(run, (m - rank) * p[i]); out[i] = min(1.0, run)
    return out


def r2_wilcoxon_min_p(n):
    """Smallest attainable two-sided exact Wilcoxon signed-rank p with n untied pairs (0.25 for n=3, 0.0625 for n=5)."""
    return 1.0 if n < 1 else min(1.0, 2.0 ** (1 - n))


def r2_paired_compare(a, b, level=0.95):
    """Paired comparison of two models over shared seeds. Difference = a - b (a = reference, e.g. the full model)."""
    a, b = np.asarray(a, float), np.asarray(b, float); d = a - b; n = len(d)
    ti = r2_t_interval(d, level)
    out = dict(n=n, mean_diff=ti["mean"], sd_diff=ti["sd"], ci_lo=ti["lo"], ci_hi=ti["hi"], dz=r2_dz(d),
               wilcoxon_min_p=r2_wilcoxon_min_p(n), p_t=float("nan"), p_wilcoxon=float("nan"), p_shapiro=float("nan"))
    if n >= 2:
        if np.ptp(d) <= 1e-12 * max(1.0, float(np.abs(d).max())):          # zero-variance differences: t is infinite
            out["p_t"] = 1.0 if abs(d[0]) <= 1e-12 else 0.0
        else:
            out["p_t"] = float(_st.ttest_rel(a, b).pvalue)
        if np.any(d != 0):
            for meth in ("exact", "auto"):
                try:
                    out["p_wilcoxon"] = float(_st.wilcoxon(a, b, method=meth).pvalue); break
                except Exception:
                    continue
    if n >= 3 and np.ptp(d) > 1e-12:
        try:
            out["p_shapiro"] = float(_st.shapiro(d).pvalue)          # almost no power below n~8: reported, not relied on
        except Exception:
            pass
    return out


def r2_mcnemar_exact(correct_a, correct_b):
    ca, cb = np.asarray(correct_a, bool), np.asarray(correct_b, bool)
    b, c = int((ca & ~cb).sum()), int((~ca & cb).sum())
    p = 1.0 if b + c == 0 else float(_st.binomtest(b, b + c, 0.5).pvalue)
    return dict(a_only=b, b_only=c, p=p)


# ── 2. cluster bootstrap (resample duplicate clusters, not images) ───────────
def r2_effective_clusters(cl, n):
    if cl is None:
        return np.arange(n, dtype=np.int64)
    cl = np.asarray(cl).astype(np.int64).copy()
    neg = cl < 0
    if neg.any():                                  # external images carry cluster = -1 -> each is its own cluster
        cl[neg] = -1 - np.arange(int(neg.sum()))
    return cl


def r2_cluster_confusions(y, pred, cl, K=4):
    y, pred = np.asarray(y, int), np.asarray(pred, int)
    cl = r2_effective_clusters(cl, len(y)); uniq, inv = np.unique(cl, return_inverse=True)
    cc = np.zeros((len(uniq), K * K)); np.add.at(cc, (inv, y * K + pred), 1)
    return cc


def r2_conf_metrics(cm):
    """cm: (B, K, K) counts, rows = true class."""
    tp = np.einsum("bii->bi", cm); n_true = cm.sum(2); n_pred = cm.sum(1)
    rec = tp / np.maximum(n_true, 1); prec = tp / np.maximum(n_pred, 1)
    f1 = 2 * prec * rec / np.maximum(prec + rec, 1e-12)
    return dict(accuracy=tp.sum(1) / np.maximum(cm.sum((1, 2)), 1), macro_f1=f1.mean(1),
                balanced_accuracy=rec.mean(1), recall=rec, precision=prec, f1=f1)


def r2_cluster_bootstrap(y, pred, cl, K=4, B=2000, seed=0, level=0.95):
    cc = r2_cluster_confusions(y, pred, cl, K); C = len(cc); rng = np.random.default_rng(seed)
    w = rng.multinomial(C, np.full(C, 1.0 / C), size=B).astype(float)
    m = r2_conf_metrics((w @ cc).reshape(B, K, K)); pt = r2_conf_metrics(cc.sum(0).reshape(1, K, K)); a = (1 - level) / 2
    out = {k: dict(point=float(pt[k][0]), lo=float(np.quantile(m[k], a)), hi=float(np.quantile(m[k], 1 - a)))
           for k in ("accuracy", "macro_f1", "balanced_accuracy")}
    out["n_clusters"] = int(C)
    return out


def r2_cluster_bootstrap_diff(correct_a, correct_b, cl, B=2000, seed=0, level=0.95):
    """Paired accuracy difference (a - b) on the SAME images, resampling clusters."""
    d = np.asarray(correct_a, float) - np.asarray(correct_b, float); n = len(d)
    cl = r2_effective_clusters(cl, n); _, inv = np.unique(cl, return_inverse=True); C = inv.max() + 1
    da = np.bincount(inv, weights=d, minlength=C); sz = np.bincount(inv, minlength=C).astype(float)
    rng = np.random.default_rng(seed); w = rng.multinomial(C, np.full(C, 1.0 / C), size=B).astype(float)
    diff = (w @ da) / np.maximum(w @ sz, 1); a = (1 - level) / 2
    p = float(min(1.0, 2 * min((diff <= 0).mean(), (diff >= 0).mean())))
    return dict(point=float(da.sum() / sz.sum()), lo=float(np.quantile(diff, a)), hi=float(np.quantile(diff, 1 - a)), p=p)


def r2_cluster_bootstrap_group_diff(correct, cl, mask_a, mask_b, B=2000, seed=0, level=0.95):
    """Accuracy(group a) - accuracy(group b) with cluster resampling (e.g. most- vs least-similar test images)."""
    correct = np.asarray(correct, float); n = len(correct); cl = r2_effective_clusters(cl, n)
    _, inv = np.unique(cl, return_inverse=True); C = inv.max() + 1
    ma, mb = np.asarray(mask_a, bool), np.asarray(mask_b, bool)
    ca = np.bincount(inv, weights=correct * ma, minlength=C); na = np.bincount(inv, weights=ma.astype(float), minlength=C)
    cb = np.bincount(inv, weights=correct * mb, minlength=C); nb = np.bincount(inv, weights=mb.astype(float), minlength=C)
    rng = np.random.default_rng(seed); w = rng.multinomial(C, np.full(C, 1.0 / C), size=B).astype(float)
    diff = (w @ ca) / np.maximum(w @ na, 1) - (w @ cb) / np.maximum(w @ nb, 1); a = (1 - level) / 2
    pt = ca.sum() / max(na.sum(), 1) - cb.sum() / max(nb.sum(), 1)
    return dict(point=float(pt), lo=float(np.quantile(diff, a)), hi=float(np.quantile(diff, 1 - a)))


# ── 3. tie-aware selective prediction ────────────────────────────────────────
def r2_risk_coverage(u, correct):
    """Selective-risk curve keeping the most-certain images first. Tied scores are treated as randomly ordered
    (expected risk), so a score with a huge tie block (saturated vacuity) is not penalised or favoured by file order."""
    u = np.asarray(u, float); correct = np.asarray(correct, bool); n = len(u)
    order = np.argsort(u, kind="mergesort"); us = np.round(u[order], 9); err = (~correct[order]).astype(float)
    new = np.r_[True, us[1:] != us[:-1]]; gid = np.cumsum(new) - 1; G = int(gid[-1]) + 1
    size = np.bincount(gid, minlength=G).astype(float); esum = np.bincount(gid, weights=err, minlength=G)
    start = np.r_[0.0, np.cumsum(size)[:-1]]; ebefore = np.r_[0.0, np.cumsum(esum)[:-1]]
    pos = np.arange(n); j = pos - start[gid] + 1
    cum = ebefore[gid] + j * (esum[gid] / size[gid])
    risk = cum / (pos + 1); cov = (pos + 1) / n
    return cov, risk, float(risk.mean())


def r2_risk_at(risk, c):
    k = max(int(math.floor(c * len(risk) + 1e-9)), 1)
    return float(risk[k - 1])


def r2_tie_stats(u, decimals=4):
    """Share of images whose score is indistinguishable at 10^-decimals (saturated vacuity differs only in the 5th decimal)."""
    u = np.asarray(u, float); u = u[np.isfinite(u)]
    if len(u) == 0:
        return dict(n_unique=0, largest_tie_frac=float("nan"))
    _, cnt = np.unique(np.round(u, decimals), return_counts=True)
    return dict(n_unique=int(len(cnt)), largest_tie_frac=float(cnt.max() / len(u)))


def r2_entropy(prob):
    K = prob.shape[1]
    return -(prob * np.log(np.clip(prob, 1e-12, 1))).sum(1) / math.log(K)


# ── 4. calibration (same definitions as the training script: 15 equal-width bins, (lo, hi]) ──
def r2_ece(prob, y, n_bins=15):
    conf = prob.max(1); acc = (prob.argmax(1) == y).astype(float); e = 0.0
    edges = np.linspace(0, 1, n_bins + 1)
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            e += m.mean() * abs(acc[m].mean() - conf[m].mean())
    return float(e)


def r2_brier(prob, y):
    return float(((prob - np.eye(prob.shape[1])[y]) ** 2).sum(1).mean())


def r2_nll(prob, y):
    return float(-np.log(np.clip(prob[np.arange(len(y)), y], 1e-12, 1)).mean())


def r2_apply_T(prob, T):
    lg = np.log(np.clip(prob, 1e-12, 1)) / T; lg = lg - lg.max(1, keepdims=True); e = np.exp(lg)
    return e / e.sum(1, keepdims=True)


def r2_fit_T(prob, y, grid=np.linspace(0.25, 6.0, 231)):
    return float(grid[int(np.argmin([r2_nll(r2_apply_T(prob, T), y) for T in grid]))])


def r2_reliability_bins(prob, y, n_bins=15, min_n=1):
    conf = prob.max(1); acc = (prob.argmax(1) == y).astype(float); edges = np.linspace(0, 1, n_bins + 1)
    xs, ys, ns = [], [], []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.sum() >= min_n:
            xs.append(conf[m].mean()); ys.append(acc[m].mean()); ns.append(int(m.sum()))
    return np.array(xs), np.array(ys), np.array(ns)


# ── 5. per-run metrics ───────────────────────────────────────────────────────
def r2_arrays(df, K=4):
    y = df["label"].values.astype(int); pred = df["pred"].values.astype(int)
    prob = df[[f"p{k}" for k in range(K)]].values.astype(float)
    vac = df["vacuity"].values.astype(float) if "vacuity" in df else np.full(len(df), np.nan)
    return y, pred, prob, vac


def r2_metrics(df, K=4):
    y, pred, prob, vac = r2_arrays(df, K)
    pr, rc, f1, sup = precision_recall_fscore_support(y, pred, labels=list(range(K)), zero_division=0)
    lb = label_binarize(y, classes=list(range(K))); aucs = []
    for k in range(K):
        if 0 < lb[:, k].sum() < len(y):
            aucs.append(roc_auc_score(lb[:, k], prob[:, k]))
    return dict(n=int(len(y)), accuracy=float((y == pred).mean()), macro_f1=float(f1.mean()),
                balanced_accuracy=float(rc.mean()), mcc=float(matthews_corrcoef(y, pred)),
                kappa=float(cohen_kappa_score(y, pred)), mean_auc=float(np.mean(aucs)) if aucs else float("nan"))


def r2_pm(m, s, scale=100.0, d=1):
    if m is None or not np.isfinite(m):
        return "n/a"
    return f"{m * scale:.{d}f}" if (s is None or not np.isfinite(s)) else f"{m * scale:.{d}f} ± {s * scale:.{d}f}"


def r2_protocol(seeds):
    """The single reporting vocabulary used in every table/figure caption (reviewer 2, comments 1 and 9)."""
    seeds = sorted(set(int(s) for s in seeds)); n = len(seeds)
    if n == 0:
        return "n/a"
    if n == 1:
        return f"single run (seed {seeds[0]})"
    rng = f"{seeds[0]}–{seeds[-1]}" if seeds == list(range(seeds[0], seeds[-1] + 1)) else ",".join(map(str, seeds))
    return f"mean ± SD over {n} seeds ({rng})"


# ── 6. run discovery / loading from disk ─────────────────────────────────────
_R2_PRED_RE = re.compile(r"^pred_(?P<variant>.+)_s(?P<seed>\d+)_(?P<set>val|test|external)\.csv$")


class R2Runs:
    """Lazy access to every saved prediction file: R.get(variant, seed, 'test'|'val'|'external'|'test_dedup')."""

    def __init__(self, out_dir, manifest=None):
        self.out_dir, self.manifest, self._c, self._m = out_dir, manifest, {}, {}
        rows = []
        for p in sorted(glob.glob(os.path.join(out_dir, "pred_*_s*_*.csv"))):
            m = _R2_PRED_RE.match(os.path.basename(p))
            if m:
                rows.append((m["variant"], int(m["seed"]), m["set"], p))
        self.index = pd.DataFrame(rows, columns=["variant", "seed", "set", "path"])

    def variants(self):
        return sorted(self.index["variant"].unique())

    def seeds(self, variant, set_="test"):
        s = "test" if set_ == "test_dedup" else set_
        got = sorted(self.index[(self.index.variant == variant) & (self.index.set == s)].seed.tolist())
        if set_ == "test_dedup":
            got = [x for x in got if "is_rep" in self.get(variant, x, "test").columns]
        return got

    def get(self, variant, seed, set_):
        if set_ == "test_dedup":
            d = self.get(variant, seed, "test")
            return d[d["is_rep"].fillna(False).astype(bool)].reset_index(drop=True)
        key = (variant, int(seed), set_)
        if key not in self._c:
            row = self.index[(self.index.variant == variant) & (self.index.seed == int(seed)) & (self.index.set == set_)]
            if row.empty:
                raise KeyError(key)
            df = pd.read_csv(row.iloc[0]["path"])
            if self.manifest is not None:
                add = [c for c in ("plane", "seq", "max_sim_train", "cluster", "is_rep") if c in self.manifest and c not in df]
                if add:
                    df = df.merge(self.manifest[["path"] + add].drop_duplicates("path"), on="path", how="left")
            self._c[key] = df
        return self._c[key]

    def metrics(self, variant, seed, set_):
        key = (variant, int(seed), set_)
        if key not in self._m:
            self._m[key] = r2_metrics(self.get(variant, seed, set_))
        return self._m[key]


def _clusters_of(df):
    return df["cluster"].values if "cluster" in df and df["cluster"].notna().all() else None


# ── 7. tables ────────────────────────────────────────────────────────────────
def r2_table_multiseed(R, sets=("test", "test_dedup", "external"),
                       metrics=("accuracy", "macro_f1", "balanced_accuracy", "mcc", "kappa", "mean_auc")):
    rows = []
    for v in R.variants():
        for s in sets:
            seeds = R.seeds(v, s)
            if not seeds:
                continue
            vals = {m: [R.metrics(v, sd, s)[m] for sd in seeds] for m in metrics}
            row = dict(variant=v, set=s, n_seeds=len(seeds), seeds=",".join(map(str, seeds)),
                       protocol=r2_protocol(seeds), n_images=R.metrics(v, seeds[0], s)["n"])
            for m in metrics:
                ti = r2_t_interval(vals[m])
                row.update({f"{m}_mean": ti["mean"], f"{m}_sd": ti["sd"], f"{m}_ci_lo": ti["lo"], f"{m}_ci_hi": ti["hi"]})
            row["accuracy_fmt"] = r2_pm(row["accuracy_mean"], row["accuracy_sd"])
            row["macro_f1_fmt"] = r2_pm(row["macro_f1_mean"], row["macro_f1_sd"], 1, 3)
            row["mcc_fmt"] = r2_pm(row["mcc_mean"], row["mcc_sd"], 1, 3)
            row["mean_auc_fmt"] = r2_pm(row["mean_auc_mean"], row["mean_auc_sd"], 1, 3)
            rows.append(row)
    return pd.DataFrame(rows)


def _aligned_correct(da, db):
    mm = da[["path", "label", "pred"]].merge(db[["path", "pred"]], on="path", suffixes=("_a", "_b"))
    return (mm.pred_a == mm.label).values, (mm.pred_b == mm.label).values, mm


def r2_table_ablation(R, base="full", sets=("test", "external"), metrics=("accuracy", "macro_f1", "mcc"),
                      ref_seed=42, B=2000, alpha=R2_ALPHA):
    """One row per (set, metric, variant): difference = base - variant over the seeds both completed. Holm is applied inside
    each (set, metric) family. 'supported' = Holm-adjusted p < alpha AND the 95% t-interval excludes 0."""
    rows = []
    for s in sets:
        for m in metrics:
            fam = []
            for v in R.variants():
                if v == base:
                    continue
                common = [sd for sd in R.seeds(base, s) if sd in R.seeds(v, s)]
                if len(common) < 2:
                    continue
                a = [R.metrics(base, sd, s)[m] for sd in common]; b = [R.metrics(v, sd, s)[m] for sd in common]
                pc = r2_paired_compare(a, b)
                row = dict(set=s, metric=m, variant=v, base=base, n_common=len(common), seeds=",".join(map(str, common)),
                           protocol=r2_protocol(common), base_mean=float(np.mean(a)), base_sd=float(np.std(a, ddof=1)),
                           var_mean=float(np.mean(b)), var_sd=float(np.std(b, ddof=1)),
                           diff_pp=100 * pc["mean_diff"], ci_lo_pp=100 * pc["ci_lo"], ci_hi_pp=100 * pc["ci_hi"], dz=pc["dz"],
                           p_t=pc["p_t"], p_wilcoxon=pc["p_wilcoxon"], wilcoxon_min_p=pc["wilcoxon_min_p"], p_shapiro=pc["p_shapiro"])
                if m == "accuracy":
                    ps = []
                    for sd in common:
                        ca, cb, _ = _aligned_correct(R.get(base, sd, s), R.get(v, sd, s))
                        ps.append(r2_mcnemar_exact(ca, cb)["p"])
                    row["mcnemar_p_by_seed"] = ";".join(f"{p:.3g}" for p in ps)
                    row["n_seeds_mcnemar_sig"] = int(sum(p < alpha for p in ps))
                    if ref_seed in common:
                        da, db = R.get(base, ref_seed, s), R.get(v, ref_seed, s)
                        ca, cb, mm = _aligned_correct(da, db)
                        cl = mm[["path"]].merge(da[["path"] + (["cluster"] if "cluster" in da else [])], on="path")
                        cb_res = r2_cluster_bootstrap_diff(ca, cb, cl["cluster"].values if "cluster" in cl else None, B=B)
                        row.update(cluster_boot_diff_pp=100 * cb_res["point"], cluster_boot_lo_pp=100 * cb_res["lo"],
                                   cluster_boot_hi_pp=100 * cb_res["hi"], cluster_boot_p=cb_res["p"])
                fam.append(row)
            holm = r2_holm([r["p_t"] for r in fam])
            for r, h in zip(fam, holm):
                r["p_holm"] = float(h); r["family_size"] = len(fam)
                sig = bool(np.isfinite(h) and h < alpha and not (r["ci_lo_pp"] <= 0 <= r["ci_hi_pp"]))
                r["supported"] = sig
                r["verdict"] = ("supported: base better" if r["diff_pp"] > 0 else "supported: variant better") if sig else "numerical only"
                r["base_fmt"] = r2_pm(r["base_mean"], r["base_sd"], 100, 1 if m == "accuracy" else 1)
                r["variant_fmt"] = r2_pm(r["var_mean"], r["var_sd"], 100, 1)
            rows += fam
    return pd.DataFrame(rows)


def r2_error_patterns(cm, names=R2_CLASSES, top=8):
    cm = np.asarray(cm); rows = []
    for i, a in enumerate(names):
        for j, b in enumerate(names):
            if i != j and cm[i, j] > 0:
                rows.append(dict(true=a, predicted=b, count=int(cm[i, j]), pct_of_true_class=100 * cm[i, j] / max(cm[i].sum(), 1)))
    return pd.DataFrame(rows).sort_values("count", ascending=False).head(top).reset_index(drop=True)


def r2_per_class(df, K=4):
    y, pred, prob, vac = r2_arrays(df, K)
    cm = confusion_matrix(y, pred, labels=list(range(K))); tp = np.diag(cm).astype(float)
    n_true, n_pred, N = cm.sum(1).astype(float), cm.sum(0).astype(float), float(cm.sum())
    fp, fn = n_pred - tp, n_true - tp; tn = N - tp - fp - fn
    prec = tp / np.maximum(n_pred, 1); rec = tp / np.maximum(n_true, 1); spec = tn / np.maximum(tn + fp, 1)
    f1 = 2 * prec * rec / np.maximum(prec + rec, 1e-12)
    return cm, dict(precision=prec, recall=rec, specificity=spec, f1=f1, support=n_true, n_pred=n_pred, tp=tp)


def r2_reweight_cm(cm, prevalence):
    """Re-weight a confusion matrix to another class mix (e.g. external -> internal prevalence). Precision depends on prevalence;
    recall and specificity-per-class-row do not."""
    cm = np.asarray(cm, float); rate = cm / np.maximum(cm.sum(1, keepdims=True), 1)
    cw = rate * np.asarray(prevalence, float)[:, None]; tp = np.diag(cw)
    prec = tp / np.maximum(cw.sum(0), 1e-12); rec = tp / np.maximum(cw.sum(1), 1e-12)
    f1 = 2 * prec * rec / np.maximum(prec + rec, 1e-12)
    return dict(accuracy=float(tp.sum()), macro_f1=float(f1.mean()), balanced_accuracy=float(rec.mean()), precision=prec, recall=rec, f1=f1)


def r2_table_external_classwise(R, variants=("full", "softmax_head"), set_="external", seed_ref=42,
                                class_names=R2_CLASSES, ref_prevalence=None, ref_variant="full"):
    K = len(class_names)
    if ref_prevalence is None:
        try:
            lab = R.get(ref_variant, R.seeds(ref_variant, "test")[0], "test")["label"].values
            ref_prevalence = np.bincount(lab, minlength=K) / len(lab)
        except Exception:
            ref_prevalence = np.full(K, 1.0 / K)
    cls_rows, over_rows, err_tables = [], [], {}
    for v in variants:
        seeds = R.seeds(v, set_)
        if not seeds:
            continue
        per, rw, over = {}, {}, {k: [] for k in ("accuracy", "macro_f1", "balanced_accuracy")}
        cms = {}
        for sd in seeds:
            cm, pc = r2_per_class(R.get(v, sd, set_), K); per[sd] = pc; cms[sd] = cm
            rw[sd] = r2_reweight_cm(cm, ref_prevalence)
            mm = r2_conf_metrics(cm[None].astype(float))
            for k in over:
                over[k].append(float(mm[k][0]))
        ref = seed_ref if seed_ref in seeds else seeds[0]
        for k, c in enumerate(class_names):
            row = dict(variant=v, set=set_, **{"class": c}, n_seeds=len(seeds), protocol=r2_protocol(seeds),
                       support=int(per[ref]["support"][k]))
            for key in ("precision", "recall", "f1", "specificity"):
                ti = r2_t_interval([per[sd][key][k] for sd in seeds]); row[f"{key}_mean"], row[f"{key}_sd"] = ti["mean"], ti["sd"]
                row[f"{key}_fmt"] = r2_pm(ti["mean"], ti["sd"])
            ti = r2_t_interval([rw[sd]["precision"][k] for sd in seeds]); row["precision_at_ref_prev_mean"] = ti["mean"]
            row[f"recall_seed{ref}"] = float(per[ref]["recall"][k]); row[f"precision_seed{ref}"] = float(per[ref]["precision"][k])
            lo, hi = r2_wilson(per[ref]["tp"][k], per[ref]["support"][k]); row[f"recall_seed{ref}_ci_lo"], row[f"recall_seed{ref}_ci_hi"] = lo, hi
            lo, hi = r2_wilson(per[ref]["tp"][k], per[ref]["n_pred"][k]); row[f"precision_seed{ref}_ci_lo"], row[f"precision_seed{ref}_ci_hi"] = lo, hi
            cls_rows.append(row)
        orow = dict(variant=v, set=set_, n_seeds=len(seeds), protocol=r2_protocol(seeds))
        for k in over:
            ti = r2_t_interval(over[k]); orow[f"{k}_mean"], orow[f"{k}_sd"] = ti["mean"], ti["sd"]
        for k in ("accuracy", "macro_f1"):
            ti = r2_t_interval([rw[sd][k] for sd in seeds]); orow[f"{k}_at_ref_prevalence_mean"], orow[f"{k}_at_ref_prevalence_sd"] = ti["mean"], ti["sd"]
        orow["ref_prevalence"] = ",".join(f"{p:.3f}" for p in ref_prevalence)
        over_rows.append(orow)
        err_tables[v] = r2_error_patterns(cms[ref], class_names)
    return pd.DataFrame(cls_rows), pd.DataFrame(over_rows), err_tables


def r2_table_calibration(R, variants=("full", "softmax_head"), sets=("test", "external")):
    rows = []
    for v in variants:
        for s in sets:
            seeds = R.seeds(v, s)
            if not seeds:
                continue
            acc = {k: [] for k in ("ece", "brier", "nll", "ece_ts", "brier_ts", "nll_ts", "T")}
            for sd in seeds:
                y, _, prob, _ = r2_arrays(R.get(v, sd, s))
                acc["ece"].append(r2_ece(prob, y)); acc["brier"].append(r2_brier(prob, y)); acc["nll"].append(r2_nll(prob, y))
                if sd in R.seeds(v, "val"):
                    yv, _, pv, _ = r2_arrays(R.get(v, sd, "val")); T = r2_fit_T(pv, yv); pt = r2_apply_T(prob, T)   # T from VALIDATION only
                    acc["ece_ts"].append(r2_ece(pt, y)); acc["brier_ts"].append(r2_brier(pt, y)); acc["nll_ts"].append(r2_nll(pt, y)); acc["T"].append(T)
            row = dict(variant=v, set=s, n_seeds=len(seeds), seeds=",".join(map(str, seeds)), protocol=r2_protocol(seeds))
            for k, vals in acc.items():
                ti = r2_t_interval(vals); row[f"{k}_mean"], row[f"{k}_sd"] = ti["mean"], ti["sd"]
            rows.append(row)
    return pd.DataFrame(rows)


def r2_table_selective(R, variants=None, sets=("test", "external"), covs=(0.9, 0.8, 0.7)):
    """Error detection and selective prediction for every uncertainty score, tie-aware, per seed then mean ± SD."""
    per = []
    for v in (variants or R.variants()):
        for s in sets:
            for sd in R.seeds(v, s):
                df = R.get(v, sd, s); y, pred, prob, vac = r2_arrays(df); correct = pred == y
                scores = {"1-maxprob": 1 - prob.max(1), "entropy": r2_entropy(prob)}
                if np.isfinite(vac).all():
                    scores = {"vacuity": vac, **scores}
                for name, u in scores.items():
                    cov, risk, aurc = r2_risk_coverage(u, correct)
                    both = 0 < (~correct).sum() < len(correct)
                    row = dict(variant=v, set=s, seed=sd, score=name, error_rate=float((~correct).mean()),
                               auroc=float(roc_auc_score(~correct, u)) if both else float("nan"),
                               auprc=float(average_precision_score(~correct, u)) if both else float("nan"), aurc=aurc)
                    row.update({f"risk_at_{int(c * 100)}": r2_risk_at(risk, c) for c in covs})
                    row.update(r2_tie_stats(u)); per.append(row)
    per = pd.DataFrame(per)
    if per.empty:
        return per, per
    agg = []
    for (v, s, sc), g in per.groupby(["variant", "set", "score"], sort=False):
        row = dict(variant=v, set=s, score=sc, n_seeds=len(g), protocol=r2_protocol(g.seed))
        for c in [c for c in g.columns if c not in ("variant", "set", "seed", "score")]:
            ti = r2_t_interval(g[c].values); row[f"{c}_mean"], row[f"{c}_sd"] = ti["mean"], ti["sd"]
        agg.append(row)
    return pd.DataFrame(agg), per


def r2_softplus(x):
    x = np.asarray(x, float)
    return np.log1p(np.exp(-np.abs(x))) + np.maximum(x, 0)


def r2_table_saturation(R, variants=None, sets=("test", "external"), clamp_by_variant=None, cap_by_variant=None,
                        default_clamp=20.0, default_cap=100.0, label_smooth=0.05, K=4):
    """How often does the evidential head sit at its ceiling?  Evidence = softplus(clamp(logit, -c, c)) (<= softplus(c)), so
    max probability <= (1+e_max)/(K+e_max) and a one-hot-evidence sample has vacuity K/(K+e_max). Reports the share of test /
    external images at those values and how many images share ONE vacuity value (ties destroy ranking)."""
    rows = []
    for v in (variants or R.variants()):
        c = (clamp_by_variant or {}).get(v, default_clamp); cap = (cap_by_variant or {}).get(v, default_cap)
        e_max = float(min(r2_softplus(c), cap)); a_cap = 1.0 + e_max
        p_ceiling, u_onehot = a_cap / (a_cap + K - 1), K / (a_cap + K - 1); target = 1 - label_smooth + label_smooth / K
        for s in sets:
            for sd in R.seeds(v, s):
                df = R.get(v, sd, s); y, pred, prob, vac = r2_arrays(df)
                if not np.isfinite(vac).all():
                    continue
                correct = pred == y; mp = prob.max(1); both = 0 < (~correct).sum() < len(correct)
                rows.append(dict(variant=v, set=s, seed=sd, clamp=c, evidence_max=e_max, prob_ceiling=p_ceiling, vacuity_onehot=u_onehot,
                                 smoothed_target=target, ceiling_below_target=bool(p_ceiling < target),
                                 frac_prob_at_ceiling=float((mp >= p_ceiling - 1e-3).mean()),
                                 frac_vacuity_within_2pct_of_onehot=float((np.abs(vac - u_onehot) <= 0.02 * u_onehot).mean()),
                                 vacuity_median=float(np.median(vac)), vacuity_q25=float(np.quantile(vac, .25)),
                                 vacuity_q75=float(np.quantile(vac, .75)), vacuity_p99=float(np.quantile(vac, .99)),
                                 auroc_vacuity=float(roc_auc_score(~correct, vac)) if both else float("nan"),
                                 auroc_1_minus_maxprob=float(roc_auc_score(~correct, 1 - mp)) if both else float("nan"),
                                 **{f"vac_{k}": val for k, val in r2_tie_stats(vac).items()}))
    per = pd.DataFrame(rows)
    if per.empty:
        return per, per
    agg = []
    for (v, s), g in per.groupby(["variant", "set"], sort=False):
        row = dict(variant=v, set=s, n_seeds=len(g), protocol=r2_protocol(g.seed))
        for c in g.columns:
            if c in ("variant", "set", "seed"):
                continue
            if g[c].dtype == bool:
                row[c] = bool(g[c].iloc[0])
            else:
                ti = r2_t_interval(g[c].values); row[f"{c}_mean"], row[f"{c}_sd"] = ti["mean"], ti["sd"]
        agg.append(row)
    return pd.DataFrame(agg), per


def r2_table_similarity(R, base="full", set_="test", q=4, seed_ref=42, B=2000):
    """Leakage sensitivity: accuracy by quantile bin of each test image's maximum cosine similarity to the TRAINING set.
    Accuracy that rises with similarity indicates near-neighbour help; a flat profile supports a small leakage effect."""
    seeds = R.seeds(base, set_)
    if not seeds:
        return None, None
    d0 = R.get(base, seeds[0], set_)
    if "max_sim_train" not in d0 or d0["max_sim_train"].isna().all():
        return None, None
    codes0 = pd.qcut(d0["max_sim_train"].rank(method="first"), q, labels=False)
    edges = {b: (float(d0.loc[codes0 == b, "max_sim_train"].min()), float(d0.loc[codes0 == b, "max_sim_train"].max())) for b in range(q)}
    rows, spear = [], []
    for sd in seeds:
        df = R.get(base, sd, set_); codes = pd.qcut(df["max_sim_train"].rank(method="first"), q, labels=False)
        corr = (df.pred == df.label).values
        spear.append(float(_st.spearmanr(df["max_sim_train"].values, corr).statistic))
        for b in range(q):
            m = (codes == b).values; rows.append(dict(seed=sd, bin=b, n=int(m.sum()), acc=float(corr[m].mean()), k=int(corr[m].sum())))
    per = pd.DataFrame(rows); out = []
    for b in range(q):
        g = per[per.bin == b]; ref = g[g.seed == (seed_ref if seed_ref in seeds else seeds[0])].iloc[0]
        lo, hi = r2_wilson(ref.k, ref.n); ti = r2_t_interval(g.acc.values)
        out.append(dict(variant=base, set=set_, bin=b + 1, sim_min=edges[b][0], sim_max=edges[b][1], n=int(ref.n),
                        acc_mean=ti["mean"], acc_sd=ti["sd"], acc_ref=float(ref.acc), acc_ref_ci_lo=lo, acc_ref_ci_hi=hi,
                        n_seeds=len(seeds), protocol=r2_protocol(seeds)))
    out = pd.DataFrame(out)
    dref = R.get(base, seed_ref if seed_ref in seeds else seeds[0], set_); codes = pd.qcut(dref["max_sim_train"].rank(method="first"), q, labels=False).values
    cb = r2_cluster_bootstrap_group_diff((dref.pred == dref.label).values, _clusters_of(dref), codes == q - 1, codes == 0, B=B)
    summ = dict(spearman_sim_vs_correct_mean=float(np.mean(spear)), spearman_sd=float(np.std(spear, ddof=1)) if len(spear) > 1 else float("nan"),
                top_minus_bottom_pp=100 * cb["point"], top_minus_bottom_lo_pp=100 * cb["lo"], top_minus_bottom_hi_pp=100 * cb["hi"])
    return out, summ


def r2_table_groups(R, by="plane", base="full", set_="test", seed_ref=42, min_n=20):
    seeds = R.seeds(base, set_)
    if not seeds:
        return None
    d0 = R.get(base, seeds[0], set_)
    if by not in d0 or d0[by].isna().all():
        return None
    rows = []
    for g in sorted(d0[by].dropna().unique()):
        accs, f1s = [], []
        for sd in seeds:
            df = R.get(base, sd, set_); m = (df[by] == g).values
            if m.sum() == 0:
                continue
            y, pred = df.label.values[m], df.pred.values[m]; accs.append(float((y == pred).mean()))
            f1s.append(float(precision_recall_fscore_support(y, pred, labels=list(range(4)), average="macro", zero_division=0)[2]))
        dref = R.get(base, seed_ref if seed_ref in seeds else seeds[0], set_); m = (dref[by] == g).values
        k = int((dref.pred.values[m] == dref.label.values[m]).sum()); lo, hi = r2_wilson(k, int(m.sum()))
        ta, tf = r2_t_interval(accs), r2_t_interval(f1s)
        rows.append(dict(variant=base, set=set_, by=by, group=g, n=int(m.sum()), acc_mean=ta["mean"], acc_sd=ta["sd"], acc_ref=k / max(m.sum(), 1),
                         acc_ref_ci_lo=lo, acc_ref_ci_hi=hi, macro_f1_mean=tf["mean"], macro_f1_sd=tf["sd"], small_group=bool(m.sum() < min_n),
                         n_seeds=len(seeds), protocol=r2_protocol(seeds)))
    out = pd.DataFrame(rows)
    try:                                                    # heterogeneity of accuracy across groups (seed_ref; images are not independent)
        dref = R.get(base, seed_ref if seed_ref in seeds else seeds[0], set_)
        ct = pd.crosstab(dref[by], dref.pred == dref.label)
        out["chi2_p_ref"] = float(_st.chi2_contingency(ct.values)[1])
    except Exception:
        pass
    return out


def r2_table_ci(R, variants=None, set_="test", seed_ref=42, B=2000):
    rows = []
    for v in (variants or R.variants()):
        seeds = R.seeds(v, set_)
        if not seeds:
            continue
        sd = seed_ref if seed_ref in seeds else seeds[0]; df = R.get(v, sd, set_); y, pred, _, _ = r2_arrays(df)
        k = int((y == pred).sum()); lo, hi = r2_wilson(k, len(y))
        img = r2_cluster_bootstrap(y, pred, None, B=B); clu = r2_cluster_bootstrap(y, pred, _clusters_of(df), B=B)
        rows.append(dict(variant=v, set=set_, seed=sd, n=len(y), n_clusters=clu["n_clusters"], accuracy=k / len(y), wilson_lo=lo, wilson_hi=hi,
                         image_boot_lo=img["accuracy"]["lo"], image_boot_hi=img["accuracy"]["hi"],
                         cluster_boot_lo=clu["accuracy"]["lo"], cluster_boot_hi=clu["accuracy"]["hi"],
                         macro_f1=clu["macro_f1"]["point"], macro_f1_cluster_lo=clu["macro_f1"]["lo"], macro_f1_cluster_hi=clu["macro_f1"]["hi"],
                         balanced_acc=clu["balanced_accuracy"]["point"], balanced_acc_cluster_lo=clu["balanced_accuracy"]["lo"],
                         balanced_acc_cluster_hi=clu["balanced_accuracy"]["hi"]))
    return pd.DataFrame(rows)


# ── 8. image-level helpers (pure numpy/scipy; the torch glue lives in the training script) ──
def r2_parse_brisc_name(path):
    """BRISC file names end in <class>_<plane>_<sequence>, e.g. brisc2025_test_00261_me_ax_t1 -> ('ax', 't1')."""
    stem = os.path.splitext(os.path.basename(str(path)))[0].lower()
    m = re.search(r"_(ax|co|sa)_([a-z0-9]+)$", stem)
    if m:
        return m.group(1), m.group(2)
    m = re.search(r"_(ax|co|sa)(?:_|$)", stem)
    return (m.group(1) if m else "unk"), "unk"


def r2_max_sim(Q, Rm, chunk=512):
    """Row-wise maximum cosine similarity of Q (n,d) to Rm (m,d); inputs are the L2-normalised thumbnails."""
    out = np.empty(len(Q), np.float32)
    for s in range(0, len(Q), chunk):
        out[s:s + chunk] = (Q[s:s + chunk] @ Rm.T).max(1)
    return out


def r2_head_mask(gray, thr=0.08, close=7):
    """Largest connected foreground blob (the head) of a grey image; analysis-only, never used in training."""
    from scipy import ndimage as ndi
    g = np.asarray(gray, float)
    if g.max() > 1.5:
        g = g / 255.0
    m = ndi.binary_fill_holes(ndi.binary_closing(g > thr, structure=np.ones((close, close))))
    lab, n = ndi.label(m)
    if n == 0:
        return m
    sizes = ndi.sum(m, lab, range(1, n + 1))
    return lab == (1 + int(np.argmax(sizes)))


def r2_cam_stats(cam, head_mask, lesion=None, corner=0.25):
    """Where does a Grad-CAM map put its mass?  frac_outside_head / frac_in_corners are shortcut indicators; pointing_hit and
    energy_ratio need a lesion mask (BRISC provides them)."""
    cam = np.clip(np.asarray(cam, float), 0, None); tot = cam.sum() + 1e-12; H, W = cam.shape
    ch, cw = max(1, int(round(corner * H))), max(1, int(round(corner * W)))
    corners = cam[:ch, :cw].sum() + cam[:ch, -cw:].sum() + cam[-ch:, :cw].sum() + cam[-ch:, -cw:].sum()
    peak = np.unravel_index(int(cam.argmax()), cam.shape)
    out = dict(frac_outside_head=float(cam[~head_mask].sum() / tot), frac_in_corners=float(corners / tot),
               corner_area_frac=float(4 * ch * cw / (H * W)), head_area_frac=float(head_mask.mean()), peak_in_head=bool(head_mask[peak]))
    if lesion is not None and lesion.any():
        e = float(cam[lesion].sum() / tot)
        out.update(pointing_hit=bool(lesion[peak]), energy_in_lesion=e, lesion_area_frac=float(lesion.mean()),
                   energy_ratio=e / max(float(lesion.mean()), 1e-12))
    return out


R2_PERTURBATIONS = ["clean", "brightness+20%", "brightness-20%", "contrast+20%", "contrast-20%", "gamma0.8", "gamma1.25",
                    "noise_sd0.03", "background_zero", "background_noise", "corners_zero"]


def r2_perturb_array(a, kind, mask=None, seed=0):
    """a: float array (H,W,3) in [0,1]. Test-time perturbations probing reliance on acquisition/background cues."""
    a = np.asarray(a, float).copy(); rng = np.random.default_rng(seed)
    if kind == "clean":
        pass
    elif kind.startswith("brightness"):
        a = a * (1.2 if "+" in kind else 0.8)
    elif kind.startswith("contrast"):
        mu = a.mean(); a = (a - mu) * (1.2 if "+" in kind else 0.8) + mu
    elif kind == "gamma0.8":
        a = np.clip(a, 0, 1) ** 0.8
    elif kind == "gamma1.25":
        a = np.clip(a, 0, 1) ** 1.25
    elif kind == "noise_sd0.03":
        a = a + rng.normal(0, 0.03, a.shape)
    elif kind in ("background_zero", "background_noise") and mask is not None:
        bg = ~mask
        a[bg] = 0.0 if kind == "background_zero" else np.clip(rng.normal(0.03, 0.02, (int(bg.sum()), a.shape[2])), 0, 1)
    elif kind == "corners_zero":
        H, W = a.shape[:2]; ch, cw = int(round(.25 * H)), int(round(.25 * W))
        a[:ch, :cw] = 0; a[:ch, -cw:] = 0; a[-ch:, :cw] = 0; a[-ch:, -cw:] = 0
    return np.clip(a, 0, 1)


def r2_perturbation_table(y, preds_by_kind, clusters=None, B=1000):
    """preds_by_kind: {kind: predicted-class array on the same images}. Accuracy, change vs clean (pp), exact McNemar, cluster-bootstrap CI."""
    y = np.asarray(y); clean = np.asarray(preds_by_kind["clean"]) == y; rows = []
    for k, p in preds_by_kind.items():
        c = np.asarray(p) == y; lo, hi = r2_wilson(int(c.sum()), len(c))
        row = dict(perturbation=k, n=len(y), accuracy=float(c.mean()), wilson_lo=lo, wilson_hi=hi, delta_vs_clean_pp=100 * float(c.mean() - clean.mean()))
        if k != "clean":
            row["mcnemar_p"] = r2_mcnemar_exact(clean, c)["p"]
            cb = r2_cluster_bootstrap_diff(c, clean, clusters, B=B); row["delta_cluster_lo_pp"], row["delta_cluster_hi_pp"] = 100 * cb["lo"], 100 * cb["hi"]
        rows.append(row)
    return pd.DataFrame(rows)


def r2_cam_summary(rows):
    """rows: DataFrame with columns branch, correct(bool) and r2_cam_stats keys -> per-branch means (and by correctness)."""
    d = pd.DataFrame(rows)
    if d.empty:
        return d
    cols = [c for c in d.columns if c not in ("branch", "path", "label", "pred", "correct")]
    out = d.groupby("branch")[cols].mean(numeric_only=True).reset_index(); out.insert(1, "group", "all"); out.insert(2, "n", d.groupby("branch").size().values)
    if "correct" in d:
        g = d.groupby(["branch", "correct"])[cols].mean(numeric_only=True).reset_index()
        g["group"] = np.where(g["correct"], "correct", "misclassified"); g["n"] = d.groupby(["branch", "correct"]).size().values
        out = pd.concat([out, g.drop(columns="correct")], ignore_index=True)
    out["branch"] = out["branch"].map(lambda b: R2_BRANCH_LABEL.get(b, b))
    return out


def r2_linear_probe(Z, y, folds=5, seed=0):
    """Can a linear probe read `y` (class, imaging plane...) from the fused embedding?  Compare against the majority-class rate."""
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import StratifiedKFold, cross_val_score
    Z, y = np.asarray(Z, float), np.asarray(y)
    Z = (Z - Z.mean(0)) / (Z.std(0) + 1e-6); _, cnt = np.unique(y, return_counts=True)
    k = int(max(2, min(folds, cnt.min())))
    sc = cross_val_score(LogisticRegression(max_iter=2000), Z, y, cv=StratifiedKFold(k, shuffle=True, random_state=seed))
    return dict(acc_mean=float(sc.mean()), acc_sd=float(sc.std(ddof=1)) if len(sc) > 1 else float("nan"), majority_rate=float(cnt.max() / cnt.sum()), folds=k)


# ── 9. figures ───────────────────────────────────────────────────────────────
_R2_COL = ["#1F6FEB", "#D9480F", "#1A7F37", "#B08800", "#8250DF", "#57606A"]


def _r2_style(ax):
    ax.set_facecolor("#F6F8FA")
    for sp in ax.spines.values():
        sp.set_edgecolor("#D0D7DE")


def r2_fig_forest(abl, path, set_="test", metric="accuracy", base="full"):
    d = abl[(abl["set"] == set_) & (abl["metric"] == metric)].sort_values("diff_pp")
    if d.empty:
        return
    fig, ax = plt.subplots(figsize=(7.5, 0.55 * len(d) + 1.8)); _r2_style(ax)
    for i, (_, r) in enumerate(d.iterrows()):
        col = _R2_COL[2] if r["supported"] else _R2_COL[5]
        ax.plot([r.ci_lo_pp, r.ci_hi_pp], [i, i], color=col, lw=2); ax.plot(r.diff_pp, i, "o", color=col)
        ax.text(ax.get_xlim()[1], i, f"  d_z={r.dz:.2f}, p_Holm={r.p_holm:.3f}", va="center", fontsize=7)
    ax.axvline(0, color="k", lw=1, ls="--"); ax.set_yticks(range(len(d))); ax.set_yticklabels(d["variant"])
    ax.set_xlabel(f"{metric}: {base} − variant (pp), 95% t-interval; green = supported (Holm p<0.05, CI excludes 0)")
    ax.set_title(f"Ablation vs {base} — {set_} — {d.iloc[0]['protocol'] if d['protocol'].nunique() == 1 else 'paired seeds per row'}", fontsize=9, fontweight="bold")
    fig.tight_layout(); fig.savefig(path, dpi=300, bbox_inches="tight"); plt.close(fig)


def r2_fig_reliability_grid(R, variants=("full", "softmax_head"), sets=("test", "external"), path="reliability_grid.png", n_bins=15, min_n=10):
    """Per-seed reliability curves (15 equal-width bins; bins with < min_n images are not drawn) with a confidence histogram strip."""
    fig, axes = plt.subplots(len(variants), len(sets), figsize=(4.8 * len(sets), 4.6 * len(variants)), squeeze=False)
    for i, v in enumerate(variants):
        for j, s in enumerate(sets):
            ax = axes[i, j]; _r2_style(ax); ax.plot([0, 1], [0, 1], "--", color="grey", lw=1); seeds = R.seeds(v, s)
            if not seeds:
                ax.set_axis_off(); continue
            eces, ece_ts, confs = [], [], []
            for sd in seeds:
                y, _, p, _ = r2_arrays(R.get(v, sd, s)); x, a, _ = r2_reliability_bins(p, y, n_bins, min_n); ax.plot(x, a, "-o", ms=2.5, color=_R2_COL[0], alpha=.45, lw=1)
                eces.append(r2_ece(p, y)); confs.append(p.max(1))
                if sd in R.seeds(v, "val"):
                    yv, _, pv, _ = r2_arrays(R.get(v, sd, "val")); pt = r2_apply_T(p, r2_fit_T(pv, yv)); x2, a2, _ = r2_reliability_bins(pt, y, n_bins, min_n)
                    ax.plot(x2, a2, "--o", ms=2.5, color=_R2_COL[1], alpha=.45, lw=1); ece_ts.append(r2_ece(pt, y))
            ax.plot([], [], color=_R2_COL[0], label=f"raw (mean ECE {np.mean(eces):.3f})")
            if ece_ts:
                ax.plot([], [], "--", color=_R2_COL[1], label=f"temperature-scaled, T from val (mean ECE {np.mean(ece_ts):.3f})")
            ax.set_title(f"{v} — {s}\n{r2_protocol(seeds)}; one line per seed", fontsize=8, fontweight="bold"); ax.set_xlabel("Confidence (max p)"); ax.set_ylabel("Accuracy")
            ax.legend(fontsize=7, loc="upper left"); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
            ins = ax.inset_axes([0, 0, 1, 0.16]); ins.hist(np.concatenate(confs), bins=n_bins, range=(0, 1), color="#8C959F"); ins.set_yscale("log"); ins.set_xticks([]); ins.tick_params(labelsize=6)
            ins.set_ylabel("n", fontsize=6)
    fig.tight_layout(); fig.savefig(path, dpi=300); plt.close(fig)


def r2_fig_uncertainty_by_class(R, variant, set_="test", seed=42, path="uncertainty_by_class.png", class_names=R2_CLASSES):
    seeds = R.seeds(variant, set_)
    if not seeds:
        return
    sd = seed if seed in seeds else seeds[0]; df = R.get(variant, sd, set_); y, pred, prob, vac = r2_arrays(df)
    scores = {"1 − max p": 1 - prob.max(1)}
    if np.isfinite(vac).all():
        scores = {"vacuity K/S": vac, **scores}
    fig, axes = plt.subplots(1, len(scores), figsize=(6.2 * len(scores), 4.4), squeeze=False)
    for ax, (name, u) in zip(axes[0], scores.items()):
        _r2_style(ax); data, pos, cols = [], [], []
        for k in range(len(class_names)):
            for c, okv in enumerate((True, False)):
                m = (y == k) & ((pred == y) == okv)
                data.append(u[m] if m.any() else np.array([np.nan])); pos.append(k * 3 + c); cols.append(_R2_COL[2] if okv else _R2_COL[1])
        bp = ax.boxplot(data, positions=pos, widths=.8, patch_artist=True, showfliers=False)
        for b, c in zip(bp["boxes"], cols):
            b.set_facecolor(c); b.set_alpha(.6)
        ax.set_xticks([k * 3 + .5 for k in range(len(class_names))]); ax.set_xticklabels(class_names); ax.set_ylabel(name)
        ax.set_title(f"{name} by true class (green = correct, red = incorrect)\n{variant}, {set_}, seed {sd}", fontsize=8, fontweight="bold")
    fig.tight_layout(); fig.savefig(path, dpi=300); plt.close(fig)


def r2_fig_risk_coverage_mean(R, variant="full", sets=("test", "external"), path="risk_coverage_mean.png"):
    grid = np.linspace(0.05, 1, 96); fig, axes = plt.subplots(1, len(sets), figsize=(5.6 * len(sets), 4.4), squeeze=False)
    for ax, s in zip(axes[0], sets):
        _r2_style(ax); seeds = R.seeds(variant, s)
        if not seeds:
            continue
        curves = {}
        for sd in seeds:
            y, pred, prob, vac = r2_arrays(R.get(variant, sd, s)); ok = pred == y
            sc = {"1 − max p": 1 - prob.max(1), "entropy": r2_entropy(prob)}
            if np.isfinite(vac).all():
                sc = {"vacuity": vac, **sc}
            for n, u in sc.items():
                cov, risk, _ = r2_risk_coverage(u, ok); curves.setdefault(n, []).append(np.interp(grid, cov, risk))
        for i, (n, cs) in enumerate(curves.items()):
            a = 100 * np.array(cs); ax.plot(grid, a.mean(0), color=_R2_COL[i], label=n)
            if len(cs) > 1:
                ax.fill_between(grid, a.mean(0) - a.std(0, ddof=1), a.mean(0) + a.std(0, ddof=1), color=_R2_COL[i], alpha=.15)
        ax.set_xlabel("Coverage"); ax.set_ylabel("Selective risk (%)"); ax.legend(fontsize=7)
        ax.set_title(f"Risk–coverage (tie-aware) — {variant}, {s}\n{r2_protocol(seeds)}", fontsize=8, fontweight="bold")
    fig.tight_layout(); fig.savefig(path, dpi=300); plt.close(fig)


def r2_fig_saturation(R, variant, set_="test", seed=42, path="saturation.png", clamp=20.0, cap=100.0, K=4):
    seeds = R.seeds(variant, set_)
    if not seeds:
        return
    sd = seed if seed in seeds else seeds[0]; y, pred, prob, vac = r2_arrays(R.get(variant, sd, set_))
    if not np.isfinite(vac).all():
        return
    e = min(r2_softplus(clamp), cap); a = 1 + e; pc, uo = a / (a + K - 1), K / (a + K - 1); ok = pred == y
    fig, ax = plt.subplots(1, 2, figsize=(10.5, 4.2)); [_r2_style(x) for x in ax]
    ax[0].hist(vac[ok], bins=60, alpha=.7, color=_R2_COL[2], label=f"correct (n={ok.sum()})"); ax[0].hist(vac[~ok], bins=60, alpha=.7, color=_R2_COL[1], label=f"incorrect (n={(~ok).sum()})")
    ax[0].axvline(uo, color="k", ls="--", lw=1, label=f"one-hot evidence at clamp: K/(K+{e:.1f}) = {uo:.4f}"); ax[0].set_yscale("log")
    ax[0].set_xlabel("Vacuity K/S"); ax[0].set_ylabel("Images (log)"); ax[0].legend(fontsize=7)
    ax[1].scatter(prob.max(1), vac, s=6, c=np.where(ok, _R2_COL[2], _R2_COL[1]), alpha=.5); ax[1].axvline(pc, color="k", ls="--", lw=1)
    ax[1].set_xlabel("max probability"); ax[1].set_ylabel("Vacuity"); ax[1].set_title(f"ceiling p = {pc:.3f}", fontsize=8)
    fig.suptitle(f"Evidence saturation — {variant}, {set_}, seed {sd} (clamp ±{clamp:g})", fontsize=9, fontweight="bold"); fig.tight_layout()
    fig.savefig(path, dpi=300); plt.close(fig)


def r2_fig_bars(df, xcol, ycol, ci_lo, ci_hi, path, title, ylabel="Accuracy"):
    if df is None or len(df) == 0:
        return
    fig, ax = plt.subplots(figsize=(1.2 * len(df) + 3, 4)); _r2_style(ax); x = np.arange(len(df))
    ax.bar(x, df[ycol] * 100, color=_R2_COL[0], alpha=.8); ax.errorbar(x, df[ycol] * 100, yerr=[100 * (df[ycol] - df[ci_lo]), 100 * (df[ci_hi] - df[ycol])], fmt="none", ecolor="k", capsize=3)
    ax.set_xticks(x); ax.set_xticklabels([str(v) for v in df[xcol]], fontsize=8); ax.set_ylabel(ylabel + " (%)"); ax.set_ylim(max(0, 100 * df[ci_lo].min() - 5), 101)
    for i, n in enumerate(df["n"]):
        ax.text(i, 100.3, f"n={n}", ha="center", fontsize=7)
    ax.set_title(title, fontsize=9, fontweight="bold"); fig.tight_layout(); fig.savefig(path, dpi=300); plt.close(fig)


def r2_pick_failures(pred_df, n=8, seed=0):
    """Misclassified test images for the failure-case Grad-CAM figure: the most frequent (true, predicted) confusions first."""
    bad = pred_df[pred_df["label"] != pred_df["pred"]]
    if bad.empty:
        return bad
    pairs = bad.groupby(["label", "pred"]).size().sort_values(ascending=False)
    take = list(pairs.index[:max(1, min(len(pairs), n // 2))]); per = int(math.ceil(n / len(take))); picks = []
    for lab, prd in take:
        g = bad[(bad["label"] == lab) & (bad["pred"] == prd)]; picks.append(g.sample(min(per, len(g)), random_state=seed))
    return pd.concat(picks).head(n).reset_index(drop=True)


def r2_fig_scatter_by(xy, groups, path, title):
    """2-D embedding coloured by an acquisition variable (imaging plane, sequence...) to check clustering by acquisition instead of class."""
    groups = np.asarray(groups); fig, ax = plt.subplots(figsize=(6.2, 5.2)); _r2_style(ax)
    for i, g in enumerate(sorted(set(groups))):
        m = groups == g; ax.scatter(xy[m, 0], xy[m, 1], s=9, color=_R2_COL[i % len(_R2_COL)], alpha=.75, label=f"{g} (n={int(m.sum())})")
    ax.legend(fontsize=8); ax.set_xlabel("dim 1"); ax.set_ylabel("dim 2"); ax.set_title(title, fontsize=9, fontweight="bold")
    fig.tight_layout(); fig.savefig(path, dpi=300); plt.close(fig)


# ── 10. formatting, registry, consistency checks, orchestrator ───────────────
def r2_format_ablation(abl):
    d = abl.copy()
    if d.empty:
        return d
    d["diff_fmt"] = d.apply(lambda r: f"{r.diff_pp:+.1f} [{r.ci_lo_pp:+.1f}, {r.ci_hi_pp:+.1f}]", axis=1)
    d["dz_fmt"] = d["dz"].map(lambda v: "n/a" if not np.isfinite(v) else f"{v:.2f}")
    d["p_t_fmt"] = d["p_t"].map(lambda v: "n/a" if not np.isfinite(v) else f"{v:.3f}")
    d["p_holm_fmt"] = d["p_holm"].map(lambda v: "n/a" if not np.isfinite(v) else f"{v:.3f}")
    return d


def r2_df_to_md(df, cols, headers=None):
    headers = headers or cols; out = ["| " + " | ".join(headers) + " |", "|" + "|".join("---" for _ in cols) + "|"]
    for _, r in df.iterrows():
        out.append("| " + " | ".join(str(r[c]) for c in cols) + " |")
    return "\n".join(out)


def r2_df_to_latex(df, cols, headers, caption, label):
    esc = lambda s: str(s).replace("_", r"\_").replace("%", r"\%").replace("±", r"$\pm$").replace("−", "-")
    lines = [r"\begin{table}[t]", r"\centering", r"\small", rf"\caption{{{esc(caption)}}}", rf"\label{{{label}}}",
             r"\begin{tabular}{" + "l" * len(cols) + "}", r"\toprule", " & ".join(esc(h) for h in headers) + r" \\", r"\midrule"]
    for _, r in df.iterrows():
        lines.append(" & ".join(esc(r[c]) for c in cols) + r" \\")
    lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
    return "\n".join(lines)


R2_CONVENTIONS = """# Reporting conventions (reviewer 2, comments 1 and 9)

Every number in the abstract, text, tables and figure captions carries exactly one of these labels:

* **single run (seed 42)**: one trained model, the pre-specified primary seed.
* **mean ± SD over n seeds (42–46)**: independent trainings that differ only in seed (initialisation, sampler, augmentation).
  The unit of analysis is the seed. Intervals are t-intervals with n-1 degrees of freedom.
* **five-fold group-aware cross-validation**: pooled BRISC images, folds grouped by near-duplicate cluster, seed 42.

Rules
1. Seeds are never mixed inside one comparison: paired tests use only the seeds both models completed, and the table says which.
2. The official BRISC test split is scored once per trained model; validation data select checkpoints and fit the temperature.
3. Seeds do NOT sample new patients. Test-set sampling uncertainty is reported separately (Wilson interval; cluster bootstrap that
   resamples near-duplicate clusters). Repeated runs on the same test images are not independent test samples.
4. A difference is called *statistically supported* only if the Holm-adjusted paired-t p < 0.05 AND its 95% t-interval excludes 0
   (Holm family = all variants compared with the reference in that set/metric). Everything else is *numerical only*.
5. With 3-5 seeds normality of paired differences cannot be verified and the exact Wilcoxon test cannot go below p = 0.25 (n=3) /
   0.0625 (n=5); both are reported as exploratory.
6. Complexity: one canonical pair of numbers (parameters, GFLOPs incl. the analytic scan estimate) from analysis_complexity.json.
"""


def _r2_load_json(p):
    try:
        with open(p) as f:
            return json.load(f)
    except Exception:
        return {}


def r2_registry(out_dir, R, tables, warnings_extra=()):
    """Canonical numbers in one place (single source of truth for the abstract, tables and captions) + consistency warnings."""
    reg = dict(conventions="reporting_conventions.md", numbers={}, warnings=list(warnings_extra))
    cx = _r2_load_json(os.path.join(out_dir, "analysis_complexity.json"))
    if cx:
        scan = cx.get("ssm_scan_elementwise_flops_G_estimate", 0.0); mm = cx.get("total_flops_G_matmul_conv", float("nan"))
        reg["numbers"]["parameters_M"] = cx.get("total_params_M"); reg["numbers"]["trainable_parameters_M"] = cx.get("trainable_params_M")
        reg["numbers"]["GFLOPs_matmul_conv"] = mm; reg["numbers"]["GFLOPs_scan_estimate"] = scan
        reg["numbers"]["GFLOPs_total_incl_scan"] = cx.get("total_flops_G_incl_scan", mm + scan)
        mods = cx.get("module_params_M", {})
        if mods and abs(sum(mods.values()) - cx.get("total_params_M", 0)) > 1e-3:
            reg["warnings"].append(f"module parameters sum to {sum(mods.values()):.3f} M but total is {cx.get('total_params_M'):.3f} M")
    ms = tables.get("multiseed")
    if ms is not None and len(ms):
        for _, r in ms.iterrows():
            reg["numbers"][f"{r.variant}|{r.set}"] = dict(protocol=r.protocol, n_seeds=int(r.n_seeds), accuracy=r.accuracy_fmt, macro_f1=r.macro_f1_fmt, mcc=r.mcc_fmt, auc=r.mean_auc_fmt)
        for v in R.variants():
            if len(R.seeds(v, "test")) < 3:
                reg["warnings"].append(f"{v}: only {len(R.seeds(v, 'test'))} seed(s) -> no interval can be trusted")
        for v in R.variants():
            if v != "full" and R.seeds("full", "test") and set(R.seeds("full", "test")) != set(R.seeds(v, "test")):
                reg["warnings"].append(f"{v} completed seeds {R.seeds(v, 'test')} vs full {R.seeds('full', 'test')}; paired tests use the {len(set(R.seeds('full', 'test')) & set(R.seeds(v, 'test')))} shared seeds only")
        sizes = {int(r.n_images) for _, r in ms[ms.set == "test"].iterrows()}
        if len(sizes) > 1:
            reg["warnings"].append(f"test sets of different size across variants: {sorted(sizes)}")
    cvp = os.path.join(out_dir, "table_cv_folds.csv")
    if os.path.exists(cvp):
        cv = pd.read_csv(cvp); reg["numbers"]["cross_validation"] = dict(protocol="five-fold group-aware cross-validation (seed 42)", n_folds=len(cv),
                                                                       accuracy=r2_pm(cv.accuracy.mean(), cv.accuracy.std(ddof=1)), macro_f1=r2_pm(cv.macro_f1.mean(), cv.macro_f1.std(ddof=1), 1, 3))
    sat = tables.get("saturation")
    if sat is not None and len(sat):
        for _, r in sat.iterrows():
            if r.get("frac_prob_at_ceiling_mean", 0) > 0.5:
                reg["warnings"].append(f"{r.variant}/{r.set}: {100 * r.frac_prob_at_ceiling_mean:.0f}% of images sit at the probability ceiling {r.prob_ceiling_mean:.3f} "
                                       f"-> vacuity has little dynamic range (say so in Sections 3.5 and 4.7)")
    return reg


def r2_summary_md(R, tables, reg):
    L = ["# Reviewer-2 analysis summary (auto-generated)", "", f"Variants found: {', '.join(R.variants())}", ""]
    ab = tables.get("ablation")
    if ab is not None and len(ab):
        L += ["## Ablation vs reference (accuracy, paired over shared seeds; Holm within family)", ""]
        d = r2_format_ablation(ab[(ab.metric == "accuracy")])
        L.append(r2_df_to_md(d, ["set", "variant", "n_common", "base_fmt", "variant_fmt", "diff_fmt", "dz_fmt", "p_t_fmt", "p_holm_fmt", "verdict"],
                             ["set", "variant", "seeds", "reference %", "variant %", "diff pp [95% CI]", "d_z", "p", "p Holm", "verdict"])); L.append("")
    sat = tables.get("saturation")
    if sat is not None and len(sat):
        L += ["## Evidence saturation", ""]
        L.append(r2_df_to_md(sat.round(4), ["variant", "set", "n_seeds", "prob_ceiling_mean", "frac_prob_at_ceiling_mean", "vacuity_median_mean", "vac_largest_tie_frac_mean",
                                           "auroc_vacuity_mean", "auroc_1_minus_maxprob_mean"])); L.append("")
    if reg["warnings"]:
        L += ["## Warnings to resolve before submission", ""] + [f"* {w}" for w in reg["warnings"]]
    return "\n".join(L)


def r2_write_checksums(out_dir, patterns=("split_manifest.csv", "run_config.json", "variants.json", "environment.txt", "duplicate_report.json",
                                          "cv_folds_manifest.csv", "pred_*.csv", "analysis_*.json")):
    sums = {}
    for pat in patterns:
        for p in sorted(glob.glob(os.path.join(out_dir, pat))):
            h = hashlib.sha256()
            with open(p, "rb") as f:
                for chunk in iter(lambda: f.read(1 << 20), b""):
                    h.update(chunk)
            sums[os.path.basename(p)] = h.hexdigest()
    with open(os.path.join(out_dir, "r2_checksums.json"), "w") as f:
        json.dump(sums, f, indent=1)
    return sums


def r2_run_all(out_dir, base="full", seed_ref=42, B=2000, subdir="r2_tables"):
    """Regenerate every reviewer-2 table/figure from the saved predictions (CPU only)."""
    import traceback
    od = os.path.join(out_dir, subdir); os.makedirs(od, exist_ok=True)
    mp = os.path.join(out_dir, "split_manifest.csv"); man = pd.read_csv(mp) if os.path.exists(mp) else None
    cfg, vcfgs = _r2_load_json(os.path.join(out_dir, "run_config.json")), _r2_load_json(os.path.join(out_dir, "variants.json"))
    d_clamp, d_cap, ls = cfg.get("evidence_clamp", 20.0), cfg.get("max_evidence", 100.0), cfg.get("label_smooth", 0.05)
    clamp_by = {v: d.get("evidence_clamp", d_clamp) for v, d in vcfgs.items()}; cap_by = {v: d.get("max_evidence", d_cap) for v, d in vcfgs.items()}
    R = R2Runs(out_dir, man)
    if R.index.empty:
        print(f"[R2] no pred_*_s*_*.csv files in {out_dir}"); return {}
    log, T = [], {}

    def step(name, fn):
        try:
            r = fn(); log.append((name, "ok")); return r
        except Exception as e:
            log.append((name, f"FAILED {type(e).__name__}: {e}")); print(f"[R2] {name} failed\n{traceback.format_exc()}"); return None

    def save(name, df):
        if df is not None and len(df):
            df.to_csv(os.path.join(od, f"{name}.csv"), index=False)

    variants = R.variants(); heads = [v for v in (base, "softmax_head") if v in variants]
    T["multiseed"] = step("multiseed", lambda: r2_table_multiseed(R)); save("table_r2_multiseed", T["multiseed"])
    T["ablation"] = step("ablation", lambda: r2_table_ablation(R, base=base, ref_seed=seed_ref, B=B))
    if T["ablation"] is not None and len(T["ablation"]):
        fa = r2_format_ablation(T["ablation"]); save("table_r2_ablation", fa)
        for s in ("test", "external"):
            step(f"forest_{s}", lambda s=s: r2_fig_forest(T["ablation"], os.path.join(od, f"fig_r2_ablation_forest_{s}.png"), s, "accuracy", base))
        a = fa[(fa.metric == "accuracy") & (fa.set == "test")]
        if len(a):
            open(os.path.join(od, "table_r2_ablation_test.tex"), "w").write(r2_df_to_latex(
                a, ["variant", "n_common", "base_fmt", "variant_fmt", "diff_fmt", "dz_fmt", "p_holm_fmt", "verdict"],
                ["Variant", "Seeds", f"{base} (\\%)", "Variant (\\%)", "Diff. pp [95\\% CI]", "$d_z$", "$p_{Holm}$", "Verdict"],
                "Ablation on the official test split; paired over shared seeds.", "tab:r2_ablation"))
    for s in ("test", "external"):
        save(f"table_r2_ci_{s}", step(f"ci_{s}", lambda s=s: r2_table_ci(R, set_=s, seed_ref=seed_ref, B=B)))
    ext = step("external_classwise", lambda: r2_table_external_classwise(R, variants=heads, seed_ref=seed_ref, ref_variant=base))
    if ext:
        save("table_r2_external_classwise", ext[0]); save("table_r2_external_overall", ext[1])
        for v, t in ext[2].items():
            save(f"table_r2_external_error_patterns_{v}", t)
    T["calibration"] = step("calibration", lambda: r2_table_calibration(R, variants=heads)); save("table_r2_calibration", T["calibration"])
    sel = step("selective", lambda: r2_table_selective(R))
    if sel:
        T["selective"] = sel[0]; save("table_r2_selective_prediction", sel[0]); save("table_r2_selective_prediction_per_seed", sel[1])
    sat = step("saturation", lambda: r2_table_saturation(R, clamp_by_variant=clamp_by, cap_by_variant=cap_by, default_clamp=d_clamp, default_cap=d_cap, label_smooth=ls))
    if sat:
        T["saturation"] = sat[0]; save("table_r2_evidence_saturation", sat[0]); save("table_r2_evidence_saturation_per_seed", sat[1])
    sim = step("similarity", lambda: r2_table_similarity(R, base=base, seed_ref=seed_ref, B=B))
    if sim and sim[0] is not None:
        save("table_r2_similarity_strata", sim[0]); json.dump(sim[1], open(os.path.join(od, "r2_similarity_summary.json"), "w"), indent=1)
        step("fig_similarity", lambda: r2_fig_bars(sim[0], "bin", "acc_ref", "acc_ref_ci_lo", "acc_ref_ci_hi", os.path.join(od, "fig_r2_similarity_strata.png"),
                                                    f"Test accuracy by similarity quartile to the training set ({base}, seed {seed_ref}; bin 4 = most similar)"))
    for by in ("plane", "seq"):
        g = step(f"groups_{by}", lambda by=by: r2_table_groups(R, by=by, base=base, seed_ref=seed_ref))
        if g is not None:
            save(f"table_r2_accuracy_by_{by}", g)
            step(f"fig_groups_{by}", lambda g=g, by=by: r2_fig_bars(g, "group", "acc_ref", "acc_ref_ci_lo", "acc_ref_ci_hi", os.path.join(od, f"fig_r2_accuracy_by_{by}.png"),
                                                                   f"Test accuracy by imaging {by} ({base}, seed {seed_ref}; Wilson 95% CI)"))
    step("fig_reliability", lambda: r2_fig_reliability_grid(R, heads, ("test", "external"), os.path.join(od, "fig_r2_reliability_grid.png")))
    for v in heads:
        for s in ("test", "external"):
            step(f"fig_unc_{v}_{s}", lambda v=v, s=s: r2_fig_uncertainty_by_class(R, v, s, seed_ref, os.path.join(od, f"fig_r2_uncertainty_by_class_{v}_{s}.png")))
            step(f"fig_sat_{v}_{s}", lambda v=v, s=s: r2_fig_saturation(R, v, s, seed_ref, os.path.join(od, f"fig_r2_saturation_{v}_{s}.png"),
                                                                       clamp_by.get(v, d_clamp), cap_by.get(v, d_cap)))
        step(f"fig_rc_{v}", lambda v=v: r2_fig_risk_coverage_mean(R, v, ("test", "external"), os.path.join(od, f"fig_r2_risk_coverage_{v}.png")))
    reg = r2_registry(out_dir, R, T)
    json.dump(reg, open(os.path.join(od, "r2_registry.json"), "w"), indent=1, default=str)
    open(os.path.join(od, "reporting_conventions.md"), "w").write(R2_CONVENTIONS)
    open(os.path.join(od, "R2_SUMMARY.md"), "w").write(r2_summary_md(R, T, reg))
    r2_write_readme(out_dir)
    r2_write_checksums(out_dir)
    bad = [f"{n}: {s}" for n, s in log if s != "ok"]
    print(f"[R2] wrote {len(os.listdir(od))} files to {od}; steps failed: {len(bad)}")
    for w in reg["warnings"]:
        print("[R2][WARN]", w)
    for b in bad:
        print("[R2][FAIL]", b)
    return T


def r2_write_readme(out_dir, variants=None, seeds_info=None):
    """REPRODUCE.md: what every output file is and the commands that regenerate it (reviewer 2, comment 6)."""
    txt = """# Reproducing the WaveSSM-TriNet results

Environment: see `environment.txt`; configuration of every run: `run_config.json`, `variants.json`; seeds: 42-46 (five-seed variants) and 42-44 (ablation variants).
Splits: `split_manifest.csv` (role, near-duplicate cluster, plane/sequence, max similarity to the training set), `cv_folds_manifest.csv` (held-out fold of every pooled BRISC image),
`duplicate_report.json` (pair counts at cosine >= threshold). Integrity: `r2_checksums.json` (SHA-256 of manifests, configs and predictions).

Training (GPU, resumable; jobs already on disk are skipped):
    R2_STAGE=heads     python wavessm_trinet_r2.py      # full + softmax_head, seeds 42-46, CV
    R2_STAGE=ablation  python wavessm_trinet_r2.py      # wavelet_only, ssm_only, cnn_only, wt_ssm_concat, ssm_cnn_concat, wt_cnn_concat, concat_mlp (seeds 42-44)
    R2_STAGE=fusion    python wavessm_trinet_r2.py      # mha_fusion, gated_fusion, concat_mlp_wide, mha_fusion_wide (param-matched), cnn_b4, no_wavelet, no_ssm (seeds 42-44)
    R2_STAGE=clamp     python wavessm_trinet_r2.py      # full_clamp100 probe of the evidence ceiling (seeds 42-44)
    (set CFG['resume_from'] / env R2_RESUME_FROM to the previous session's output folder to continue)

Analysis only (CPU; regenerates every table and figure from the saved predictions):
    python r2_analysis.py --out <folder with pred_*.csv>
    or  R2_EVAL_ONLY=1 python wavessm_trinet_r2.py

Outputs in `r2_tables/`: table_r2_multiseed, table_r2_ablation(+.tex), table_r2_ci_*, table_r2_external_classwise/overall/error_patterns_*, table_r2_calibration,
table_r2_selective_prediction, table_r2_evidence_saturation, table_r2_similarity_strata, table_r2_accuracy_by_plane/seq, fig_r2_* , r2_registry.json (canonical numbers + warnings),
reporting_conventions.md, R2_SUMMARY.md. Torch-side analyses: analysis_r2_gradcam_stats/summary.csv, fig_gradcam_failure_cases.png, analysis_r2_perturbation.csv,
analysis_r2_embedding_probe.json, fig_tsne_by_plane.png.
"""
    open(os.path.join(out_dir, "REPRODUCE.md"), "w").write(txt)



# ══════════════════════════════════════════════════════════════════════════════
#  EVALUATION UTILITIES  (merged from eval_utils.py -- no torch dependency)
# ══════════════════════════════════════════════════════════════════════════════
CLASS_NAMES = ["glioma", "meningioma", "notumor", "pituitary"]
PALETTE = ["#1F6FEB", "#D9480F", "#1A7F37", "#B08800", "#8250DF"]


# ══════════════════════════════════════════════════════════════════════════════
#  1. DATA HYGIENE: near-duplicate detection, clusters, group-aware splits
#     (Reviewer comments 2, 3, 4, 7)
# ══════════════════════════════════════════════════════════════════════════════
def thumbnail_matrix(paths, size=32, cache=None):
    """Mean-centred, L2-normalised grey thumbnails (N, size*size). Cosine = dot product."""
    from PIL import Image
    if cache and os.path.exists(cache):
        arr = np.load(cache)
        if arr.shape[0] == len(paths):
            return arr
    out = np.zeros((len(paths), size * size), dtype=np.float32)
    for i, p in enumerate(paths):
        with Image.open(p) as im:
            a = np.asarray(im.convert("L").resize((size, size), Image.BILINEAR), dtype=np.float32).ravel()
        a = a - a.mean()
        n = np.linalg.norm(a)
        out[i] = a / n if n > 1e-6 else 0.0
    if cache:
        np.save(cache, out)
    return out


def cosine_pairs(A, B=None, thr=0.985, chunk=1024):
    """All index pairs with cosine >= thr. If B is None, pairs (i<j) inside A."""
    same = B is None
    B = A if same else B
    ii_all, jj_all, ss_all = [], [], []
    for s in range(0, len(A), chunk):
        sim = A[s:s + chunk] @ B.T
        ii, jj = np.nonzero(sim >= thr)
        gi = ii + s
        keep = jj > gi if same else np.ones_like(jj, dtype=bool)
        ii_all.append(gi[keep]); jj_all.append(jj[keep]); ss_all.append(sim[ii, jj][keep])
    if not ii_all:
        return np.zeros((0, 3))
    return np.stack([np.concatenate(ii_all), np.concatenate(jj_all), np.concatenate(ss_all)], 1) \
        if ii_all else np.zeros((0, 3))


def cluster_from_pairs(n, pairs):
    """Union-find. Returns an integer cluster id per item (near-duplicate groups)."""
    parent = list(range(n))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    for i, j, _ in pairs:
        ri, rj = find(int(i)), find(int(j))
        if ri != rj:
            parent[rj] = ri
    roots = np.array([find(i) for i in range(n)])
    _, cl = np.unique(roots, return_inverse=True)
    return cl


def group_stratified_split(labels, groups, val_frac=0.1, seed=42):
    """One stratified, group-aware split. Returns (train_idx, val_idx)."""
    n_splits = max(2, int(round(1.0 / val_frac)))
    labels, groups = np.asarray(labels), np.asarray(groups)
    min_groups = min(len(np.unique(groups[labels == c])) for c in np.unique(labels))   # robust to tiny / unbalanced data
    n_splits = max(2, min(n_splits, min_groups))
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    tr, va = next(sgkf.split(np.zeros(len(labels)), labels, groups))
    return tr, va


def image_intensity_stats(paths):
    """Cheap acquisition-shift descriptors (comment 25): mean, contrast, background fraction."""
    from PIL import Image
    rows = []
    for p in paths:
        with Image.open(p) as im:
            a = np.asarray(im.convert("L").resize((128, 128)), dtype=np.float32) / 255.0
        fg = a[a > 0.05]
        rows.append(dict(path=p, mean=float(a.mean()), std=float(a.std()),
                         bg_frac=float((a <= 0.05).mean()),
                         fg_mean=float(fg.mean()) if fg.size else 0.0,
                         p95=float(np.percentile(a, 95))))
    return pd.DataFrame(rows)


# ══════════════════════════════════════════════════════════════════════════════
#  2. PREDICTION-FILE HELPERS
# ══════════════════════════════════════════════════════════════════════════════
def arrays_from_df(df, K=4):
    y = df["label"].values.astype(int)
    pred = df["pred"].values.astype(int)
    prob = df[[f"p{k}" for k in range(K)]].values.astype(np.float64)
    vac = df["vacuity"].values.astype(np.float64) if "vacuity" in df else np.full(len(df), np.nan)
    return y, pred, prob, vac


def predictive_entropy(prob):
    K = prob.shape[1]
    return -(prob * np.log(np.clip(prob, 1e-12, 1))).sum(1) / math.log(K)


# ══════════════════════════════════════════════════════════════════════════════
#  3. CLASSIFICATION METRICS + CONFIDENCE INTERVALS   (comments 5, 6, 21)
# ══════════════════════════════════════════════════════════════════════════════
def wilson_ci(k, n, z=1.96):
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n
    d = 1 + z * z / n
    c = p + z * z / (2 * n)
    a = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return ((c - a) / d, (c + a) / d)


def mean_auc_ovr(y, prob):
    K = prob.shape[1]
    lb = label_binarize(y, classes=list(range(K)))
    out = []
    for k in range(K):
        if lb[:, k].sum() in (0, len(y)):
            out.append(float("nan")); continue
        out.append(roc_auc_score(lb[:, k], prob[:, k]))
    return out


def classification_metrics(y, pred, prob, class_names=CLASS_NAMES):
    K = len(class_names)
    pr, rc, f1, sup = precision_recall_fscore_support(y, pred, labels=list(range(K)), zero_division=0)
    _, _, wf1, _ = precision_recall_fscore_support(y, pred, labels=list(range(K)), average="weighted", zero_division=0)
    per_auc = mean_auc_ovr(y, prob)
    k_ok = int((y == pred).sum()); n = len(y)
    lo, hi = wilson_ci(k_ok, n)
    return dict(
        n=n, correct=k_ok, accuracy=k_ok / n, accuracy_ci95=[lo, hi],
        macro_precision=float(pr.mean()), macro_recall=float(rc.mean()),
        macro_f1=float(f1.mean()), weighted_f1=float(wf1),
        mcc=float(matthews_corrcoef(y, pred)), kappa=float(cohen_kappa_score(y, pred)),
        mean_auc=float(np.nanmean(per_auc)),
        per_class={c: dict(precision=float(pr[i]), recall=float(rc[i]), f1=float(f1[i]),
                           support=int(sup[i]), auc=float(per_auc[i]))
                   for i, c in enumerate(class_names)},
        confusion_matrix=confusion_matrix(y, pred, labels=list(range(K))).tolist())


def bootstrap_ci(y, pred, prob, n_boot=1000, seed=0):
    """Percentile bootstrap CIs for accuracy, macro-F1 and mean AUC (substitute for DeLong)."""
    rng = np.random.RandomState(seed)
    accs, f1s, aucs = [], [], []
    n = len(y)
    for _ in range(n_boot):
        idx = rng.randint(0, n, n)
        yy, pp, pr = y[idx], pred[idx], prob[idx]
        accs.append((yy == pp).mean())
        f1s.append(precision_recall_fscore_support(yy, pp, average="macro", zero_division=0)[2])
        aucs.append(np.nanmean(mean_auc_ovr(yy, pr)))
    q = lambda v: [float(np.nanpercentile(v, 2.5)), float(np.nanpercentile(v, 97.5))]
    return dict(accuracy=q(accs), macro_f1=q(f1s), mean_auc=q(aucs))


# ══════════════════════════════════════════════════════════════════════════════
#  4. CALIBRATION  (comment 16)
# ══════════════════════════════════════════════════════════════════════════════
def ece_score(prob, y, n_bins=15):
    conf = prob.max(1); pred = prob.argmax(1); acc = (pred == y).astype(float)
    edges = np.linspace(0, 1, n_bins + 1)
    e = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            e += m.mean() * abs(acc[m].mean() - conf[m].mean())
    return float(e)


def brier_score(prob, y):
    K = prob.shape[1]
    oh = np.eye(K)[y]
    return float(((prob - oh) ** 2).sum(1).mean())


def nll_score(prob, y):
    return float(-np.log(np.clip(prob[np.arange(len(y)), y], 1e-12, 1)).mean())


def reliability_curve(prob, y, n_bins=15):
    conf = prob.max(1); acc = (prob.argmax(1) == y).astype(float)
    edges = np.linspace(0, 1, n_bins + 1)
    xs, ys, ns = [], [], []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            xs.append(conf[m].mean()); ys.append(acc[m].mean()); ns.append(int(m.sum()))
    return np.array(xs), np.array(ys), np.array(ns)


def apply_temperature(prob, T):
    lg = np.log(np.clip(prob, 1e-12, 1)) / T
    lg = lg - lg.max(1, keepdims=True)
    e = np.exp(lg)
    return e / e.sum(1, keepdims=True)


def fit_temperature(prob, y, grid=np.linspace(0.25, 6.0, 231)):
    """Temperature scaling on log-probabilities (works for softmax and Dirichlet-mean probs).
    MUST be fitted on the validation split, never on the test split."""
    nlls = [nll_score(apply_temperature(prob, T), y) for T in grid]
    return float(grid[int(np.argmin(nlls))])


def calibration_report(prob, y, n_bins=15):
    return dict(ece=ece_score(prob, y, n_bins), brier=brier_score(prob, y), nll=nll_score(prob, y))


# ══════════════════════════════════════════════════════════════════════════════
#  5. UNCERTAINTY QUALITY  (comments 17, 18, 25)
# ══════════════════════════════════════════════════════════════════════════════
def risk_coverage(uncertainty, correct):
    """Selective-risk curve: keep the most-certain fraction, report error rate among kept."""
    order = np.argsort(uncertainty, kind="mergesort")          # most certain first
    err = (~correct[order]).astype(float)
    n = len(err)
    cum_err = np.cumsum(err)
    cov = np.arange(1, n + 1) / n
    risk = cum_err / np.arange(1, n + 1)
    return cov, risk, float(risk.mean())                         # AURC = mean selective risk


def error_detection(y, pred, prob, vac):
    correct = (y == pred)
    scores = {"1-maxprob": 1 - prob.max(1), "entropy": predictive_entropy(prob)}
    if np.isfinite(vac).any():
        scores = {"vacuity": vac, **scores}
    out = {}
    for name, s in scores.items():
        if correct.all() or (~correct).all():
            out[name] = dict(auroc=float("nan"), auprc=float("nan"), aurc=float("nan"))
            continue
        cov, risk, aurc = risk_coverage(s, correct)
        out[name] = dict(auroc=float(roc_auc_score(~correct, s)),
                         auprc=float(average_precision_score(~correct, s)),
                         aurc=aurc,
                         risk_at_90cov=float(risk[int(0.9 * len(risk)) - 1]),
                         risk_at_80cov=float(risk[int(0.8 * len(risk)) - 1]))
    out["_error_rate"] = float((~correct).mean())
    return out


def high_confidence_errors(df, K=4, conf_thr=0.85):
    """Table for comment 18: confident mistakes and where each uncertainty score ranks them."""
    y, pred, prob, vac = arrays_from_df(df, K)
    mp = prob.max(1); ent = predictive_entropy(prob)
    rk = lambda v: pd.Series(v).rank(pct=True).values        # percentile; high = flagged uncertain
    t = pd.DataFrame(dict(path=df["path"].values, label=y, pred=pred, maxprob=mp, entropy=ent,
                          vacuity=vac, entropy_pctile=rk(ent), maxprob_pctile=rk(-mp),
                          vacuity_pctile=rk(vac) if np.isfinite(vac).any() else np.nan))
    return t[(y != pred) & (mp >= conf_thr)].sort_values("maxprob", ascending=False)


# ══════════════════════════════════════════════════════════════════════════════
#  6. STATISTICS ACROSS SEEDS / PAIRED TESTS  (comment 21)
# ══════════════════════════════════════════════════════════════════════════════
def summarize_seeds(vals):
    v = np.asarray(vals, dtype=float)
    n = len(v)
    m = float(v.mean()); sd = float(v.std(ddof=1)) if n > 1 else float("nan")
    ci = float(stats.t.ppf(0.975, n - 1) * sd / math.sqrt(n)) if n > 1 else float("nan")
    return dict(n=n, mean=m, sd=sd, ci95=[m - ci, m + ci])


def paired_seed_tests(a, b):
    """a, b: per-seed metric values of two variants (same seeds). Low power at n=5; say so."""
    a, b = np.asarray(a, float), np.asarray(b, float)
    out = dict(mean_diff=float((a - b).mean()))
    if len(a) >= 2 and np.any(a != b):
        out["paired_t_p"] = float(stats.ttest_rel(a, b).pvalue)
        try:
            out["wilcoxon_p"] = float(stats.wilcoxon(a, b).pvalue)
        except ValueError:
            out["wilcoxon_p"] = float("nan")
    return out


def mcnemar_exact(correct_a, correct_b):
    b = int((correct_a & ~correct_b).sum()); c = int((~correct_a & correct_b).sum())
    p = 1.0 if b + c == 0 else float(stats.binomtest(b, b + c, 0.5).pvalue)
    return dict(a_only_correct=b, b_only_correct=c, p=p)


def shapley_values(values, players):
    """values: {frozenset(subset): v(subset)}; exact Shapley over all coalitions."""
    n = len(players)
    phi = {}
    for p in players:
        others = [q for q in players if q != p]
        tot = 0.0
        for r in range(len(others) + 1):
            for S in combinations(others, r):
                S = frozenset(S)
                w = math.factorial(len(S)) * math.factorial(n - len(S) - 1) / math.factorial(n)
                tot += w * (values[S | {p}] - values[S])
        phi[p] = tot
    return phi


# ══════════════════════════════════════════════════════════════════════════════
#  7. FIGURES FROM PREDICTION FILES  (comments 5, 6, 16, 17, 24, Q2a)
# ══════════════════════════════════════════════════════════════════════════════
def _ax_style(ax):
    ax.set_facecolor("#F6F8FA")
    for sp in ax.spines.values():
        sp.set_edgecolor("#D0D7DE")


def fig_confusion(cm, class_names, path, title="Confusion matrix"):
    cm = np.array(cm); cmn = cm / cm.sum(1, keepdims=True).clip(min=1)
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(cmn, cmap="Blues", vmin=0, vmax=1, annot=cm, fmt="d", cbar_kws={"label": "Row-normalised"},
                xticklabels=class_names, yticklabels=class_names, ax=ax, linewidths=0.6, linecolor="#D0D7DE")
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(title, fontweight="bold")
    fig.tight_layout(); fig.savefig(path, dpi=300); plt.close(fig)


def fig_roc(y, prob, class_names, path, title="ROC"):
    lb = label_binarize(y, classes=list(range(len(class_names))))
    fig, ax = plt.subplots(figsize=(7, 6)); _ax_style(ax)
    for k, c in enumerate(class_names):
        if lb[:, k].sum() in (0, len(y)):
            continue
        fpr, tpr, _ = roc_curve(lb[:, k], prob[:, k])
        ax.plot(fpr, tpr, color=PALETTE[k], lw=2, label=f"{c} (AUC={auc(fpr, tpr):.4f})")
    ax.plot([0, 1], [0, 1], "--", color="grey", lw=1)
    ax.set_xlabel("FPR"); ax.set_ylabel("TPR"); ax.set_title(title, fontweight="bold"); ax.legend(loc="lower right")
    fig.tight_layout(); fig.savefig(path, dpi=300); plt.close(fig)


def fig_per_class(metrics, path):
    names = list(metrics["per_class"].keys())
    x = np.arange(len(names)); w = 0.27
    fig, ax = plt.subplots(figsize=(8, 4.5)); _ax_style(ax)
    for i, key in enumerate(["precision", "recall", "f1"]):
        ax.bar(x + (i - 1) * w, [100 * metrics["per_class"][c][key] for c in names], w, label=key, color=PALETTE[i])
    ax.set_xticks(x); ax.set_xticklabels(names); ax.set_ylim(80, 101); ax.set_ylabel("Score (%)"); ax.legend(loc="lower right")
    ax.set_title("Per-class precision / recall / F1 (from saved predictions)", fontweight="bold")
    fig.tight_layout(); fig.savefig(path, dpi=300); plt.close(fig)


def fig_reliability(prob_raw, prob_ts, y, path, n_bins=15, title="Reliability"):
    fig, ax = plt.subplots(figsize=(5.5, 5.5)); _ax_style(ax)
    ax.plot([0, 1], [0, 1], "--", color="grey")
    for pr, lab, col in [(prob_raw, "raw", PALETTE[0]), (prob_ts, "temp-scaled", PALETTE[1])]:
        if pr is None:
            continue
        x, a, _ = reliability_curve(pr, y, n_bins)
        ax.plot(x, a, "o-", color=col, label=f"{lab} (ECE={ece_score(pr, y, n_bins):.3f})")
    ax.set_xlabel("Confidence"); ax.set_ylabel("Accuracy"); ax.set_title(title, fontweight="bold"); ax.legend()
    fig.tight_layout(); fig.savefig(path, dpi=300); plt.close(fig)


def fig_risk_coverage(y, pred, prob, vac, path):
    correct = (y == pred)
    scores = {"1-maxprob": 1 - prob.max(1), "entropy": predictive_entropy(prob)}
    if np.isfinite(vac).any():
        scores = {"vacuity": vac, **scores}
    fig, ax = plt.subplots(figsize=(6, 4.5)); _ax_style(ax)
    for i, (n, s) in enumerate(scores.items()):
        cov, risk, aurc = risk_coverage(s, correct)
        ax.plot(cov, 100 * risk, color=PALETTE[i], label=f"{n} (AURC={100 * aurc:.2f}%)")
    ax.set_xlabel("Coverage"); ax.set_ylabel("Selective risk (%)"); ax.legend()
    ax.set_title("Risk–coverage", fontweight="bold")
    fig.tight_layout(); fig.savefig(path, dpi=300); plt.close(fig)


def fig_uncertainty_hist(y, pred, vac, path):
    if not np.isfinite(vac).any():
        return
    correct = (y == pred)
    fig, ax = plt.subplots(figsize=(6, 4.5)); _ax_style(ax)
    bins = np.linspace(np.nanmin(vac), np.nanmax(vac), 40)
    ax.hist(vac[correct], bins=bins, alpha=0.7, color="#1A7F37", label=f"correct (n={correct.sum()})", density=True)
    if (~correct).any():
        ax.hist(vac[~correct], bins=bins, alpha=0.7, color="#D9480F", label=f"incorrect (n={(~correct).sum()})", density=True)
    ax.set_xlabel("Vacuity (K/S)"); ax.set_ylabel("Density"); ax.legend()
    ax.set_title("Vacuity: correct vs incorrect (density-normalised)", fontweight="bold")
    fig.tight_layout(); fig.savefig(path, dpi=300); plt.close(fig)


def fig_embedding(Z, y, class_names, path, title):
    fig, ax = plt.subplots(figsize=(7, 6)); _ax_style(ax)
    for k, c in enumerate(class_names):
        m = y == k
        ax.scatter(Z[m, 0], Z[m, 1], s=10, color=PALETTE[k], alpha=0.8, label=f"{c} (n={m.sum()})")
    ax.set_title(title, fontweight="bold"); ax.legend(); ax.set_xlabel("dim 1"); ax.set_ylabel("dim 2")
    fig.tight_layout(); fig.savefig(path, dpi=300); plt.close(fig)


# ══════════════════════════════════════════════════════════════════════════════
#  8. ONE-CALL EVALUATION OF A PREDICTION FILE
# ══════════════════════════════════════════════════════════════════════════════
def evaluate_predictions(df, out_prefix, class_names=CLASS_NAMES, val_df=None, subset_mask=None,
                         bootstrap=True, make_figs=True):
    """Everything the reviewer asked for, from one CSV. Returns a JSON-serialisable dict."""
    K = len(class_names)
    os.makedirs(os.path.dirname(out_prefix) or ".", exist_ok=True)
    if subset_mask is not None:
        df = df[subset_mask].reset_index(drop=True)
    y, pred, prob, vac = arrays_from_df(df, K)
    res = dict(metrics=classification_metrics(y, pred, prob, class_names))
    if bootstrap:
        res["bootstrap_ci95"] = bootstrap_ci(y, pred, prob)
    res["calibration_raw"] = calibration_report(prob, y)
    prob_ts = None
    if val_df is not None and len(val_df):
        yv, _, pv, _ = arrays_from_df(val_df, K)
        T = fit_temperature(pv, yv)                       # fitted on VALIDATION only
        prob_ts = apply_temperature(prob, T)
        res["temperature"] = T
        res["calibration_temp_scaled"] = calibration_report(prob_ts, y)
    res["error_detection"] = error_detection(y, pred, prob, vac)
    # per-class calibration / uncertainty (external no-tumor analysis, comment 25)
    res["per_class_uncertainty"] = {}
    for k, c in enumerate(class_names):
        m = y == k
        if m.any():
            res["per_class_uncertainty"][c] = dict(
                n=int(m.sum()), recall=float((pred[m] == k).mean()),
                mean_vacuity=float(np.nanmean(vac[m])) if np.isfinite(vac[m]).any() else None,
                mean_entropy=float(predictive_entropy(prob[m]).mean()),
                mean_maxprob=float(prob[m].max(1).mean()),
                ece=ece_score(prob[m], y[m]) if m.sum() >= 30 else None)
    with open(out_prefix + "_summary.json", "w") as f:
        json.dump(res, f, indent=2)
    high_confidence_errors(df, K).to_csv(out_prefix + "_high_conf_errors.csv", index=False)
    if make_figs:
        fig_confusion(res["metrics"]["confusion_matrix"], class_names, out_prefix + "_confusion.png")
        fig_roc(y, prob, class_names, out_prefix + "_roc.png")
        fig_per_class(res["metrics"], out_prefix + "_perclass.png")
        fig_reliability(prob, prob_ts, y, out_prefix + "_reliability.png")
        fig_risk_coverage(y, pred, prob, vac, out_prefix + "_risk_coverage.png")
        fig_uncertainty_hist(y, pred, vac, out_prefix + "_vacuity_hist.png")
    return res


def sanity_check_pred_df(df, K=4, tol=1e-3, name=""):
    """Fail loudly if probabilities / predictions are inconsistent (guards against comment-5 type errors)."""
    prob = df[[f"p{k}" for k in range(K)]].values.astype(np.float64)
    bad = ~np.isfinite(prob).all(1)
    if bad.any():
        raise AssertionError(f"[{name}] {int(bad.sum())}/{len(prob)} rows have NaN/inf probabilities -> the model produced "
                             f"non-finite outputs (fp16 overflow or diverged training). See NaN notes in the code.")
    sums = prob.sum(1)
    assert np.allclose(sums, 1.0, atol=tol), f"[{name}] probabilities do not sum to 1 (min={sums.min():.4f}, max={sums.max():.4f})"
    assert (prob.argmax(1) == df["pred"].values).all(), f"[{name}] pred column != argmax(prob)"
    return True


# ══════════════════════════════════════════════════════════════════════════════
#  CONFIG  (everything the reviewer asked to be documented lives here -> run_config.json)
# ══════════════════════════════════════════════════════════════════════════════
CFG = dict(
    data_root="/kaggle/input/datasets/briscdataset/brisc2025/brisc2025/classification_task",
    ext_data_root="/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset",
    ext_splits=["Training"],            # which Nickparvar folders form the external set (log the counts!)
    mask_root=None,                     # optional: folder with BRISC test masks for Grad-CAM hit-rate
    out_dir="/kaggle/working",
    img_size=224, num_classes=4, class_names=CLASS_NAMES,

    # training protocol (matches the notebook that produced the original numbers)
    epochs=50, batch_size=32, grad_accum=2, num_workers=2,
    lr=3e-4, weight_decay=1e-4, label_smooth=0.05, ema_decay=0.999, clip_grad=1.0,
    pct_start=0.15, div_factor=25.0, final_div_factor=1e4,
    backbone="efficientnet_b3", n_trainable_backbone_tensors=60,
    pretrained=True,                    # if True and weights cannot be loaded -> raise (no silent fallback)

    # architecture
    ssm_d_model=192, ssm_d_state=16, ssm_layers=3, ssm_patch=16,
    branch_dim=192, fusion_dim=384, n_tokens=49, attn_heads=4, dropout=0.25,

    # evidential loss
    lambda_kl=0.05, kl_anneal_epochs=10, max_evidence=100.0,
    evidence_clamp=20.0,                # logits are clamped to +-evidence_clamp before softplus -> evidence <= softplus(clamp) (R2-5)

    # data hygiene
    val_frac=0.10, split_seed=42, dup_thr=0.985, drop_train_dups_of_test=True, drop_external_dups=True,

    # experiments
    seeds=[42, 43, 44, 45, 46],
    ablation_seeds=[42, 43, 44],         # R2-2: every non-headline variant is trained on these seeds
    five_seed_variants=["full", "softmax_head"],   # R2-5: heads comparison needs the same five seeds for both
    variants=["full", "softmax_head",                                                     # heads comparison (5 seeds)
              "wavelet_only", "ssm_only", "cnn_only", "wt_ssm_concat", "ssm_cnn_concat", "wt_cnn_concat", "concat_mlp",   # Table-11 ablation (3 seeds)
              "mha_fusion", "gated_fusion", "concat_mlp_wide", "mha_fusion_wide", "cnn_b4", "no_wavelet", "no_ssm",       # alternative fusion / capacity baselines
              "full_clamp100"],                                                           # evidence-ceiling probe
    primary_variant="full",
    resume_from=None,                    # previous session's output folder (Kaggle: attach it as an input) -> ckpts/predictions are copied in
    r2_failure_cases=8, r2_gradcam_per_class=100, r2_perturb=True,
    run_cv=True, cv_folds=5, cv_epochs=None,   # None -> same epochs as main run
    run_analysis=True, tsne_per_class=250, tta=True,
)

VARIANTS = {
    "full":         dict(fusion="tcafg",  head="evidential", branches=("wt", "ssm", "cnn")),
    "softmax_head": dict(fusion="tcafg",  head="softmax",    branches=("wt", "ssm", "cnn")),
    "no_smooth":    dict(fusion="tcafg",  head="evidential", branches=("wt", "ssm", "cnn"), label_smooth=0.0),
    "concat_mlp":   dict(fusion="concat", head="evidential", branches=("wt", "ssm", "cnn")),
    "mha_fusion":   dict(fusion="mha",    head="evidential", branches=("wt", "ssm", "cnn")),
    "gated_fusion": dict(fusion="gated",  head="evidential", branches=("wt", "ssm", "cnn")),
    "cnn_only":     dict(fusion="pool",   head="evidential", branches=("cnn",)),
    "cnn_b4":       dict(fusion="pool",   head="evidential", branches=("cnn",), backbone="efficientnet_b4"),
    "no_wavelet":   dict(fusion="tcafg",  head="evidential", branches=("ssm", "cnn")),
    "no_ssm":       dict(fusion="tcafg",  head="evidential", branches=("wt", "cnn")),
    # R2-5 probe: with clamp 20 the evidence cannot exceed softplus(20)=20 -> max p = 21/24 = 0.875 < the smoothed target 0.9625.
    # clamp 100 lifts the ceiling to 101/104 = 0.971, so the vacuity/confidence ranking can be compared with and without saturation.
    "full_clamp100": dict(fusion="tcafg", head="evidential", branches=("wt", "ssm", "cnn"), evidence_clamp=100.0, max_evidence=100.0),
    # R2-2: the rows of the manuscript's single-run ablation table, now trainable over seeds (evidential head everywhere except softmax_head)
    "wavelet_only":   dict(fusion="pool",   head="evidential", branches=("wt",)),
    "ssm_only":       dict(fusion="pool",   head="evidential", branches=("ssm",)),
    "wt_ssm_concat":  dict(fusion="concat", head="evidential", branches=("wt", "ssm")),
    "ssm_cnn_concat": dict(fusion="concat", head="evidential", branches=("ssm", "cnn")),
    "wt_cnn_concat":  dict(fusion="concat", head="evidential", branches=("wt", "cnn")),
    # parameter-matched fusion baselines (TCAFG = 2.55 M parameters): hidden width / feed-forward width chosen analytically
    #   concat MLP params = 961*h + 2304  -> h = 2652 -> 2.551 M ;  2-layer MHA fusion = 374k + 770*ff -> ff = 2824 -> 2.548 M
    "concat_mlp_wide": dict(fusion="concat", head="evidential", branches=("wt", "ssm", "cnn"), fusion_hidden=2652),
    "mha_fusion_wide": dict(fusion="mha",    head="evidential", branches=("wt", "ssm", "cnn"), fusion_ff=2824),
}

RUN_MODE = "smoke" if SMOKE else os.environ.get("RUN_MODE", "paper")      # "smoke" | "quick" | "paper"
if RUN_MODE == "quick":      # 2 variants x 3 seeds x 30 epochs, no CV
    CFG.update(seeds=[42, 43, 44], epochs=30, run_cv=False, variants=["full", "softmax_head"])
elif RUN_MODE == "smoke":    # tiny synthetic run to verify the whole pipeline end-to-end (CPU is fine)
    CFG.update(data_root="/tmp/smoke/brisc/classification_task", ext_data_root="/tmp/smoke/ext",
               out_dir="/tmp/smoke/out", epochs=2, cv_epochs=1, seeds=[42, 43], batch_size=4, grad_accum=1,
               num_workers=0, pretrained=False, variants=["full", "softmax_head", "full_clamp100"],
               tsne_per_class=8, cv_folds=2, n_trainable_backbone_tensors=10**6, img_size=112)
elif RUN_MODE != "paper":
    raise ValueError(f"RUN_MODE must be smoke | quick | paper, got {RUN_MODE!r}")
CFG.setdefault("time_budget_h", 11.5)         # everything (training + analysis) must end before this many hours after start
CFG.setdefault("analysis_reserve_min", 60)      # minutes kept free at the end for aggregation + analysis figures
CFG["resume"] = True                 # skip runs whose checkpoint + test predictions already exist in out_dir
if os.environ.get("R2_RESUME_FROM"): CFG["resume_from"] = os.environ["R2_RESUME_FROM"]
R2_STAGE = os.environ.get("R2_STAGE", "all")        # heads | ablation | clamp | all
if RUN_MODE == "paper" and R2_STAGE != "all":
    CFG["variants"] = {"heads": ["full", "softmax_head"],
                       "ablation": ["wavelet_only", "ssm_only", "cnn_only", "wt_ssm_concat", "ssm_cnn_concat", "wt_cnn_concat", "concat_mlp"],
                       "fusion": ["mha_fusion", "gated_fusion", "concat_mlp_wide", "mha_fusion_wide", "cnn_b4", "no_wavelet", "no_ssm"],
                       "clamp": ["full_clamp100"]}[R2_STAGE]


def seeds_for(variant):
    """Five seeds for the headline pair (full, softmax_head); ablation_seeds for every other variant."""
    if variant in CFG["five_seed_variants"]: return list(CFG["seeds"])
    return [s for s in CFG["ablation_seeds"] if s in CFG["seeds"]] or list(CFG["seeds"][:3])


def _variants_on_disk():
    """Requested variants plus any variant that already has predictions in out_dir (earlier Kaggle sessions)."""
    return [v for v in VARIANTS if v in CFG["variants"] or
            any(os.path.exists(os.path.join(CFG["out_dir"], f"pred_{v}_s{s}_test.csv")) for s in CFG["seeds"])]
T0 = float(os.environ.get("WAVESSM_T0", time.time()))            # wall-clock start, shared with the worker processes
print(f"[CONFIG] RUN_MODE={RUN_MODE}  seeds={CFG['seeds']}  epochs={CFG['epochs']}  variants={CFG['variants']}  cv={CFG['run_cv']}")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda" and CFG.get("amp", True)
os.makedirs(CFG["out_dir"], exist_ok=True)
K = CFG["num_classes"]


def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False


def seed_worker(_):
    s = torch.initial_seed() % 2 ** 32
    np.random.seed(s); random.seed(s)


def vcfg(variant, key):
    """Variant-specific override of a CFG key."""
    return VARIANTS[variant].get(key, CFG[key])


# ══════════════════════════════════════════════════════════════════════════════
#  1. DATA: discovery, duplicate detection, group-aware splits
# ══════════════════════════════════════════════════════════════════════════════
NORM = {"notumor": "notumor", "no_tumor": "notumor", "no tumor": "notumor", "normal": "notumor",
        "glioma": "glioma", "meningioma": "meningioma", "pituitary": "pituitary"}
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}


def scan_split(split_dir):
    """Return DataFrame(path,label) for a folder with one sub-folder per class."""
    rows = []
    for d in sorted(os.listdir(split_dir)):
        full = os.path.join(split_dir, d)
        canon = NORM.get(d.lower())
        if not os.path.isdir(full) or canon not in CFG["class_names"]:
            continue
        for fn in sorted(os.listdir(full)):
            if os.path.splitext(fn)[1].lower() in IMG_EXT:
                rows.append((os.path.join(full, fn), CFG["class_names"].index(canon)))
    return pd.DataFrame(rows, columns=["path", "label"])


def find_dir(root, names):
    for n in names:
        p = os.path.join(root, n)
        if os.path.isdir(p):
            return p
    return None


def build_data():
    """Builds the master manifest. Roles: train / val / test / dropped_dup_of_test /
    external / external_dropped_dup."""
    root = CFG["data_root"]
    if not os.path.isdir(root):
        raise FileNotFoundError(f"BRISC root not found: {root}")
    tr_dir = find_dir(root, ["train", "Training", "Train", "TRAIN"])
    te_dir = find_dir(root, ["test", "Testing", "Test", "TEST"])
    if tr_dir is None or te_dir is None:
        raise FileNotFoundError(f"Could not find train/test folders under {root}")
    tr, te = scan_split(tr_dir), scan_split(te_dir)
    tr["origin"], te["origin"] = "brisc_train", "brisc_test"
    br = pd.concat([tr, te], ignore_index=True)
    print(f"[DATA] BRISC train={len(tr)}  test={len(te)}  class counts test={te.label.value_counts().sort_index().tolist()}")

    # ---- near-duplicate detection inside BRISC (comments 3, 4) -------------------
    cache = os.path.join(CFG["out_dir"], "brisc_thumbs.npy")
    T = thumbnail_matrix(br.path.tolist(), cache=cache)
    pairs = cosine_pairs(T, thr=CFG["dup_thr"])
    br["cluster"] = cluster_from_pairs(len(br), pairs)
    org = br.origin.values
    n_tt = sum((org[int(i)] == "brisc_train") != (org[int(j)] == "brisc_train") for i, j, _ in pairs)
    n_in_test = sum(org[int(i)] == "brisc_test" and org[int(j)] == "brisc_test" for i, j, _ in pairs)
    n_in_train = sum(org[int(i)] == "brisc_train" and org[int(j)] == "brisc_train" for i, j, _ in pairs)
    print(f"[DUP] near-duplicate pairs (cos>={CFG['dup_thr']}): train-train={n_in_train}  test-test={n_in_test}  train-test={n_tt}")
    dup_report = dict(train_train=int(n_in_train), test_test=int(n_in_test), train_test=int(n_tt), thr=CFG["dup_thr"])

    br["role"] = "train_pool"
    br.loc[br.origin == "brisc_test", "role"] = "test"
    test_clusters = set(br.loc[br.origin == "brisc_test", "cluster"])
    leak = (br.origin == "brisc_train") & br.cluster.isin(test_clusters)
    if CFG["drop_train_dups_of_test"]:
        br.loc[leak, "role"] = "dropped_dup_of_test"
    print(f"[DUP] train images sharing a cluster with a test image: {int(leak.sum())} "
          f"({'dropped' if CFG['drop_train_dups_of_test'] else 'kept'})")
    dup_report["train_dropped"] = int(leak.sum()) if CFG["drop_train_dups_of_test"] else 0

    # one representative per cluster inside the test set -> 'deduplicated test' metrics
    br["is_rep"] = False
    te_idx = br.index[br.role == "test"]
    first = br.loc[te_idx].groupby("cluster").head(1).index
    br.loc[first, "is_rep"] = True
    dup_report["test_unique"] = int(br.is_rep.sum())

    # ---- group-aware stratified train/val split (checkpoint-selection set) -------
    pool = br[br.role == "train_pool"]
    tr_i, va_i = group_stratified_split(pool.label.values, pool.cluster.values, CFG["val_frac"], CFG["split_seed"])
    br.loc[pool.index[va_i], "role"] = "val"
    br.loc[pool.index[tr_i], "role"] = "train"

    # ---- R2-7 / R2-8: acquisition metadata from the file names + similarity of each test image to the TRAINING set --------
    pl = [r2_parse_brisc_name(p) for p in br.path]
    br["plane"], br["seq"] = [a for a, _ in pl], [b for _, b in pl]
    br["max_sim_train"] = np.nan
    tr_idx, te_idx2 = br.index[br.role == "train"].to_numpy(), br.index[br.role == "test"].to_numpy()
    br.loc[te_idx2, "max_sim_train"] = r2_max_sim(T[te_idx2], T[tr_idx])
    print(f"[R2] planes in the test split: {br.loc[te_idx2, 'plane'].value_counts().to_dict()}   "
          f"max cosine similarity test->train: median {np.nanmedian(br.loc[te_idx2, 'max_sim_train']):.3f}")

    # ---- external cohort (comment 2) --------------------------------------------
    ext = None
    if CFG["ext_data_root"] and os.path.isdir(CFG["ext_data_root"]):
        parts = []
        for sp in CFG["ext_splits"]:
            d = find_dir(CFG["ext_data_root"], [sp, sp.lower(), sp.capitalize()])
            if d is not None:
                parts.append(scan_split(d).assign(origin=f"external_{sp}"))
        if parts:
            ext = pd.concat(parts, ignore_index=True)
            print(f"[EXT] {len(ext)} images  per-class counts (VERIFY against the paper's 5,600): "
                  f"{ext.label.value_counts().sort_index().tolist()}")
            Te = thumbnail_matrix(ext.path.tolist(), cache=os.path.join(CFG["out_dir"], "ext_thumbs.npy"))
            cross = cosine_pairs(Te, T, thr=CFG["dup_thr"])
            dup_ext = np.unique(cross[:, 0].astype(int)) if len(cross) else np.array([], int)
            ext["cluster"] = -1; ext["is_rep"] = True
            ext["role"] = "external"
            ext["plane"], ext["seq"] = "ext", "ext"
            ext["max_sim_train"] = r2_max_sim(Te, T[tr_idx])
            if CFG["drop_external_dups"] and len(dup_ext):
                ext.loc[dup_ext, "role"] = "external_dropped_dup"
            print(f"[EXT] images near-duplicating a BRISC image: {len(dup_ext)} "
                  f"({'dropped' if CFG['drop_external_dups'] else 'kept'})")
            dup_report["external_total"] = int(len(ext)); dup_report["external_dup_of_brisc"] = int(len(dup_ext))
    manifest = pd.concat([br, ext], ignore_index=True) if ext is not None else br
    r2_verify_resume_manifest(manifest)
    manifest.to_csv(os.path.join(CFG["out_dir"], "split_manifest.csv"), index=False)
    json.dump(dup_report, open(os.path.join(CFG["out_dir"], "duplicate_report.json"), "w"), indent=2)
    summary = manifest.groupby(["role", "label"]).size().unstack(fill_value=0)
    summary.columns = [CFG["class_names"][c] for c in summary.columns]
    summary.to_csv(os.path.join(CFG["out_dir"], "dataset_distribution_table.csv"))
    print(summary)
    return manifest


def build_transforms(train):
    mean, std, sz = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225], CFG["img_size"]
    if train:
        return transforms.Compose([
            transforms.Resize((sz + 32, sz + 32)), transforms.RandomCrop(sz), transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(12), transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
            transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)), transforms.RandomGrayscale(p=0.05),
            transforms.ToTensor(), transforms.Normalize(mean, std), transforms.RandomErasing(p=0.2, scale=(0.02, 0.10))])
    return transforms.Compose([transforms.Resize((sz, sz)), transforms.ToTensor(), transforms.Normalize(mean, std)])


class ImgDataset(Dataset):
    def __init__(self, df, train):
        self.paths, self.labels = df.path.tolist(), df.label.tolist()
        self.tf = build_transforms(train)

    def __len__(self): return len(self.paths)

    def __getitem__(self, i):
        with Image.open(self.paths[i]) as im:
            x = self.tf(im.convert("RGB"))
        return x, self.labels[i], i


def make_loader(df, train, seed=0, bs=None):
    ds = ImgDataset(df, train)
    bs = bs or CFG["batch_size"]
    g = torch.Generator(); g.manual_seed(seed)
    if train:
        cnt = np.bincount(ds.labels, minlength=K).clip(min=1)
        w = (1.0 / cnt)[np.array(ds.labels)]
        sampler = WeightedRandomSampler(torch.from_numpy(w).float(), len(w), replacement=True, generator=g)
        return DataLoader(ds, bs, sampler=sampler, num_workers=CFG["num_workers"], pin_memory=USE_AMP,
                          worker_init_fn=seed_worker, generator=g, drop_last=len(ds) > bs)
    return DataLoader(ds, bs, shuffle=False, num_workers=CFG["num_workers"], pin_memory=USE_AMP, worker_init_fn=seed_worker)


# ══════════════════════════════════════════════════════════════════════════════
#  2. MODEL
# ══════════════════════════════════════════════════════════════════════════════
class HaarDWT(nn.Module):
    """Single-level separable 2-D Haar DWT as fixed stride-2 depthwise 2x2 convolutions.
    LL = h(x)h, LH = g(rows)h(cols) [horizontal detail], HL = h(rows)g(cols) [vertical detail], HH = g g,
    with h=[1,1]/sqrt2, g=[1,-1]/sqrt2.  (12 x 112 x 112 for a 3 x 224 x 224 input; no padding: even size.)"""
    def __init__(self, c=3):
        super().__init__()
        h = torch.tensor([1.0, 1.0]) / math.sqrt(2); g = torch.tensor([1.0, -1.0]) / math.sqrt(2)
        k = torch.stack([torch.outer(h, h), torch.outer(g, h), torch.outer(h, g), torch.outer(g, g)]).unsqueeze(1)
        self.register_buffer("weight", k.repeat(c, 1, 1, 1)); self.c = c

    def forward(self, x):
        return F.conv2d(x.float(), self.weight, stride=2, groups=self.c).to(x.dtype)


def test_haar_vs_pywt():
    """Unit test (comment 8): LL identical; detail bands equal up to sign to pywt 'haar'."""
    try:
        import pywt
    except ImportError:
        print("[TEST] pywt not installed -> skipped"); return
    x = torch.randn(1, 1, 16, 16); m = HaarDWT(1)
    out = m(x)[0].numpy(); cA, (cH, cV, cD) = pywt.dwt2(x[0, 0].numpy(), "haar")
    ok = [np.allclose(out[0], cA, atol=1e-5)] + [np.allclose(np.abs(out[i + 1]), np.abs(b), atol=1e-5)
                                                  for i, b in enumerate([cH, cV, cD])]
    print(f"[TEST] Haar DWT vs PyWavelets (LL, LH, HL, HH): {ok}")
    assert all(ok), "DWT mismatch"


class WaveletBranch(nn.Module):
    def __init__(self, out_dim=192, c=3):
        super().__init__()
        self.dwt = HaarDWT(c); ch = 4 * c
        self.gate = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(ch, 2 * ch), nn.ReLU(True),
                                  nn.Linear(2 * ch, ch), nn.Sigmoid())
        self.encoder = nn.Sequential(
            nn.Conv2d(ch, 64, 3, padding=1, bias=False), nn.BatchNorm2d(64), nn.GELU(),
            nn.Conv2d(64, 128, 3, padding=1, stride=2, bias=False), nn.BatchNorm2d(128), nn.GELU(),
            nn.Conv2d(128, 256, 3, padding=1, stride=2, bias=False), nn.BatchNorm2d(256), nn.GELU())
        self.proj = nn.Linear(256, out_dim); self.norm = nn.LayerNorm(out_dim)

    def forward(self, x):
        wt = self.dwt(x); wt = wt * self.gate(wt)[:, :, None, None]
        fmap = self.encoder(wt)                                        # (B,256,28,28)
        pooled = F.adaptive_avg_pool2d(fmap, 7).flatten(2).transpose(1, 2)   # (B,49,256)
        return self.norm(self.proj(pooled)), fmap


def parallel_scan(dA, dBx):
    """Inclusive scan of h_t = dA_t * h_{t-1} + dBx_t (Hillis-Steele, O(L log L))."""
    B, L, D, S = dA.shape
    a, b, stride = dA, dBx, 1
    while stride < L:
        na = a[:, stride:] * a[:, :-stride]
        nb = a[:, stride:] * b[:, :-stride] + b[:, stride:]
        a = torch.cat([a[:, :stride], na], 1); b = torch.cat([b[:, :stride], nb], 1); stride *= 2
    return b


class SelectiveSSM(nn.Module):
    """Diagonal selective SSM. Discretisation actually used (matches Eqs. 2-4 in the revised paper):
       A = -exp(A_log) (<0), Abar = exp(Delta*A) elementwise, Bbar ~= mean_d(Delta) * B  (first-order)."""
    def __init__(self, d_model, d_state=16):
        super().__init__()
        self.d_state = d_state
        self.in_proj = nn.Linear(d_model, 2 * d_model)
        self.conv1d = nn.Conv1d(d_model, d_model, 3, padding=1, groups=d_model)
        self.x_proj = nn.Linear(d_model, 2 * d_state + 1)
        self.dt_proj = nn.Linear(1, d_model)
        self.A_log = nn.Parameter(torch.log(torch.arange(1, d_state + 1, dtype=torch.float32)).unsqueeze(0).expand(d_model, -1).clone())
        self.D = nn.Parameter(torch.ones(d_model))
        self.out_proj = nn.Linear(d_model, d_model); self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        with torch.autocast("cuda", enabled=False):                    # whole SSM in fp32: the scan must not overflow in fp16
            return self._forward(x.float())

    def _forward(self, x):
        S, res = self.d_state, x
        x_, z = self.in_proj(x).chunk(2, -1)
        x_c = F.silu(self.conv1d(x_.transpose(1, 2)).transpose(1, 2))
        p = self.x_proj(x_c)
        B_, C_ = p[..., :S], p[..., S:2 * S]
        delta = F.softplus(self.dt_proj(p[..., -1:])) + 1e-4
        A = -torch.exp(self.A_log.float())
        dA = torch.exp(delta.float().unsqueeze(-1) * A[None, None])
        dB = delta.mean(-1, keepdim=True).float() * B_.float()
        dBx = x_c.float().unsqueeze(-1) * dB.unsqueeze(2)
        h = parallel_scan(dA, dBx)
        y = (h * C_.float().unsqueeze(2)).sum(-1).to(x_c.dtype) + self.D * x_c
        return self.norm(self.out_proj(y * F.silu(z)) + res)


class SSMBlock(nn.Module):
    def __init__(self, d, s):
        super().__init__()
        self.n1 = nn.LayerNorm(d); self.ssm = SelectiveSSM(d, s); self.n2 = nn.LayerNorm(d)
        self.ff = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Dropout(0.1), nn.Linear(4 * d, d))

    def forward(self, x):
        x = self.ssm(self.n1(x)) + x
        return self.ff(self.n2(x)) + x


class SSMBranch(nn.Module):
    """Operates on RAW RGB 16x16 patches (196 tokens), unidirectional raster scan."""
    def __init__(self, img=224, patch=16, d=192, s=16, layers=3, out_dim=192):
        super().__init__()
        self.g = img // patch
        self.unfold = nn.Unfold(patch, stride=patch)
        self.patch_proj = nn.Linear(3 * patch * patch, d)
        self.pos = nn.Parameter(torch.randn(1, self.g ** 2, d) * 0.02)
        self.blocks = nn.ModuleList([SSMBlock(d, s) for _ in range(layers)])
        self.norm = nn.LayerNorm(d); self.proj = nn.Linear(d, out_dim); self.out_norm = nn.LayerNorm(out_dim)

    def forward(self, x):
        t = self.patch_proj(self.unfold(x).transpose(1, 2)) + self.pos
        for b in self.blocks:
            t = checkpoint(b, t, use_reentrant=False) if (self.training and torch.is_grad_enabled()) else b(t)   # saves ~3 GB
        t = self.norm(t)                                               # (B,196,d)
        fmap = t.transpose(1, 2).reshape(t.size(0), -1, self.g, self.g)
        pooled = F.adaptive_avg_pool2d(fmap, 7).flatten(2).transpose(1, 2)   # (B,49,d)
        return self.out_norm(self.proj(pooled)), fmap


class SpatialBranch(nn.Module):
    def __init__(self, backbone="efficientnet_b3", out_dim=192, dropout=0.25):
        super().__init__()
        try:
            self.backbone = timm.create_model(backbone, pretrained=CFG["pretrained"], num_classes=0, global_pool="")
        except Exception as e:
            raise RuntimeError(f"Could not create {backbone} (pretrained={CFG['pretrained']}): {e}. "
                               "Enable Internet / attach the weights; silent fallback is disabled (comment 27).")
        self.pretrained_loaded = bool(CFG["pretrained"])
        params = list(self.backbone.parameters())
        for p in params[:-CFG["n_trainable_backbone_tensors"]] if CFG["n_trainable_backbone_tensors"] < len(params) else []:
            p.requires_grad_(False)
        self.proj = nn.Sequential(nn.Linear(self.backbone.num_features, out_dim), nn.LayerNorm(out_dim), nn.GELU(), nn.Dropout(dropout))

    def forward(self, x):
        fmap = self.backbone.forward_features(x)                       # (B,C,7,7)
        t = F.adaptive_avg_pool2d(fmap, 7).flatten(2).transpose(1, 2)  # (B,49,C)
        return self.proj(t), fmap


# ---- fusion modules (all take dict name->(B,N,D) tokens, return (B,fusion_dim)) --------------
class DirectionalCrossAttn(nn.Module):
    """Tokens of branch i (queries) attend to tokens of branch j (keys/values); gated, then pooled."""
    def __init__(self, dim, heads, dropout):
        super().__init__()
        self.nq, self.nkv = nn.LayerNorm(dim), nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, heads, dropout=dropout * 0.4, batch_first=True)
        self.gate = nn.Sequential(nn.Linear(2 * dim, dim), nn.Sigmoid())

    def forward(self, q, kv, need_weights=False):
        kvn = self.nkv(kv)
        out, w = self.attn(self.nq(q), kvn, kvn, need_weights=need_weights, average_attn_weights=True)
        y = q + out                                                    # (B,Nq,D)
        a = y.mean(1)
        g = self.gate(torch.cat([q.mean(1), a], -1))                   # pair-specific gate
        return g * a, w


class TCAFG(nn.Module):
    """Triple Cross-Attention Fusion Gate over token sequences. For branches (w,s,c): six directed
    attentions (w<-s, w<-c, s<-w, s<-c, c<-w, c<-s); each output has its own sigmoid gate."""
    def __init__(self, names, dim=192, out_dim=384, heads=4, n_tokens=49, dropout=0.25):
        super().__init__()
        self.names = list(names)
        self.pairs = [(a, b) for a in names for b in names if a != b]
        self.pos = nn.ParameterDict({n: nn.Parameter(torch.randn(1, n_tokens, dim) * 0.02) for n in names})
        self.ca = nn.ModuleDict({f"{a}_{b}": DirectionalCrossAttn(dim, heads, dropout) for a, b in self.pairs})
        f = dim * len(self.pairs)
        self.mlp = nn.Sequential(nn.LayerNorm(f), nn.Linear(f, 2 * out_dim), nn.GELU(), nn.Dropout(dropout),
                                 nn.Linear(2 * out_dim, out_dim), nn.LayerNorm(out_dim))

    def forward(self, toks, return_attn=False):
        t = {n: toks[n] + self.pos[n] for n in self.names}
        outs, attn = [], {}
        for a, b in self.pairs:
            o, w = self.ca[f"{a}_{b}"](t[a], t[b], need_weights=return_attn)
            outs.append(o)
            if return_attn: attn[f"{a}<-{b}"] = w
        z = self.mlp(torch.cat(outs, -1))
        return (z, attn) if return_attn else z


class ConcatMLPFusion(nn.Module):
    def __init__(self, names, dim=192, out_dim=384, dropout=0.25, hidden=None, **_):
        super().__init__(); self.names = list(names)
        f = dim * len(names); h = int(hidden or 2 * out_dim)
        self.mlp = nn.Sequential(nn.LayerNorm(f), nn.Linear(f, h), nn.GELU(), nn.Dropout(dropout),
                                 nn.Linear(h, out_dim), nn.LayerNorm(out_dim))

    def forward(self, toks, return_attn=False):
        z = self.mlp(torch.cat([toks[n].mean(1) for n in self.names], -1))
        return (z, {}) if return_attn else z


class MHAFusion(nn.Module):
    """Standard multi-head self-attention fusion (CLS + all tokens, 2 transformer layers)."""
    def __init__(self, names, dim=192, out_dim=384, heads=4, n_tokens=49, dropout=0.25, ff=None, **_):
        super().__init__(); self.names = list(names)
        self.cls = nn.Parameter(torch.zeros(1, 1, dim))
        self.type_emb = nn.ParameterDict({n: nn.Parameter(torch.randn(1, 1, dim) * 0.02) for n in names})
        layer = nn.TransformerEncoderLayer(dim, heads, dim_feedforward=int(ff or 2 * dim), dropout=dropout * 0.4,
                                           batch_first=True, norm_first=True, activation="gelu")
        self.enc = nn.TransformerEncoder(layer, 2)
        self.out = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, out_dim), nn.LayerNorm(out_dim))

    def forward(self, toks, return_attn=False):
        seq = torch.cat([self.cls.expand(toks[self.names[0]].size(0), -1, -1)] +
                        [toks[n] + self.type_emb[n] for n in self.names], 1)
        z = self.out(self.enc(seq)[:, 0])
        return (z, {}) if return_attn else z


class GatedFusion(nn.Module):
    def __init__(self, names, dim=192, out_dim=384, dropout=0.25, **_):
        super().__init__(); self.names = list(names)
        self.gate = nn.Linear(dim * len(names), dim * len(names))
        f = dim * len(names)
        self.mlp = nn.Sequential(nn.LayerNorm(f), nn.Linear(f, 2 * out_dim), nn.GELU(), nn.Dropout(dropout),
                                 nn.Linear(2 * out_dim, out_dim), nn.LayerNorm(out_dim))

    def forward(self, toks, return_attn=False):
        v = torch.cat([toks[n].mean(1) for n in self.names], -1)
        z = self.mlp(v * torch.sigmoid(self.gate(v)))
        return (z, {}) if return_attn else z


class PoolFusion(nn.Module):
    """Single-branch baseline (EfficientNet-only): mean-pool tokens, project to fusion_dim."""
    def __init__(self, names, dim=192, out_dim=384, **_):
        super().__init__(); self.names = list(names)
        self.out = nn.Sequential(nn.Linear(dim * len(names), out_dim), nn.LayerNorm(out_dim))

    def forward(self, toks, return_attn=False):
        z = self.out(torch.cat([toks[n].mean(1) for n in self.names], -1))
        return (z, {}) if return_attn else z


FUSIONS = dict(tcafg=TCAFG, concat=ConcatMLPFusion, mha=MHAFusion, gated=GatedFusion, pool=PoolFusion)


class Head(nn.Module):
    """3-layer MLP 384->512->256->K (329,476 params for K=4). kind: evidential | softmax."""
    def __init__(self, in_dim, kind, dropout=0.25, clamp=20.0, cap=100.0):
        super().__init__(); self.kind = kind; self.clamp, self.cap = float(clamp), float(cap)
        self.mlp = nn.Sequential(nn.Linear(in_dim, 512), nn.GELU(), nn.Dropout(dropout),
                                 nn.Linear(512, 256), nn.GELU(), nn.Dropout(dropout / 2), nn.Linear(256, K))

    def forward(self, x):
        logits = self.mlp(x).float()
        if self.kind == "evidential":
            ev = F.softplus(logits.clamp(-self.clamp, self.clamp)).clamp(max=self.cap)      # evidence <= softplus(clamp): the source of the 0.875 ceiling at clamp 20
            alpha = ev + 1.0
            S = alpha.sum(1, keepdim=True)
            return dict(prob=alpha / S, alpha=alpha, vac=(K / S).squeeze(1), logits=logits)
        prob = F.softmax(logits, 1)
        ent = -(prob * torch.log(prob.clamp_min(1e-12))).sum(1) / math.log(K)
        return dict(prob=prob, alpha=None, vac=ent * float("nan"), logits=logits)   # vacuity undefined


class WaveSSMTriNet(nn.Module):
    def __init__(self, variant="full"):
        super().__init__()
        v = VARIANTS[variant]; B = CFG["branch_dim"]
        self.variant, self.branches = variant, tuple(v["branches"])
        if "wt" in self.branches: self.wavelet_branch = WaveletBranch(B)
        if "ssm" in self.branches:
            self.ssm_branch = SSMBranch(CFG["img_size"], CFG["ssm_patch"], CFG["ssm_d_model"], CFG["ssm_d_state"], CFG["ssm_layers"], B)
        if "cnn" in self.branches:
            self.cnn_branch = SpatialBranch(v.get("backbone", CFG["backbone"]), B, CFG["dropout"])
        fkw = dict(dim=B, out_dim=CFG["fusion_dim"], heads=CFG["attn_heads"], n_tokens=CFG["n_tokens"], dropout=CFG["dropout"])
        if "fusion_hidden" in v: fkw["hidden"] = v["fusion_hidden"]
        if "fusion_ff" in v: fkw["ff"] = v["fusion_ff"]
        self.fusion = FUSIONS[v["fusion"]](self.branches, **fkw)
        self.head = Head(CFG["fusion_dim"], v["head"], CFG["dropout"], clamp=vcfg(variant, "evidence_clamp"), cap=vcfg(variant, "max_evidence"))
        for n in self.branches:                                         # training-mean tokens for occlusion (#22)
            self.register_buffer(f"mean_{n}", torch.zeros(1, CFG["n_tokens"], B))
        self.capture, self._maps = False, {}

    def branch_tokens(self, x):
        toks, maps = {}, {}
        for n, mod in (("wt", "wavelet_branch"), ("ssm", "ssm_branch"), ("cnn", "cnn_branch")):
            if n in self.branches:
                toks[n], maps[n] = getattr(self, mod)(x)
        return toks, maps

    def forward(self, x, drop=(), return_attn=False):
        toks, maps = self.branch_tokens(x)
        for n in drop:                                                   # occlusion: replace by training-mean tokens
            if n in toks: toks[n] = getattr(self, f"mean_{n}").expand_as(toks[n]).to(toks[n].dtype)
        if self.capture:
            for m in maps.values():
                if m.requires_grad: m.retain_grad()
            self._maps = maps
        fz = self.fusion(toks, return_attn=return_attn)
        fused, attn = fz if return_attn else (fz, None)
        out = self.head(fused); out["fused"] = fused
        if return_attn: out["attn"] = attn
        return out


# ══════════════════════════════════════════════════════════════════════════════
#  3. LOSSES  (comment 15)
# ══════════════════════════════════════════════════════════════════════════════
def kl_dirichlet_to_uniform(alpha):
    K_ = alpha.shape[1]; a0 = alpha.sum(1)
    return (torch.lgamma(a0) - torch.lgamma(alpha).sum(1) - math.lgamma(K_)
            + ((alpha - 1) * (torch.digamma(alpha) - torch.digamma(a0.unsqueeze(1)))).sum(1)).mean()


class EvidentialLoss(nn.Module):
    """Expected squared error (Brier-type) + KL(Dir(alpha_tilde) || Dir(1)); NOT expected cross-entropy.
    alpha_tilde removes target-class evidence using the HARD one-hot label (smoothed label is used only in the MSE term)."""
    def __init__(self, smooth):
        super().__init__(); self.smooth = smooth

    def forward(self, out, y, epoch):
        with torch.autocast("cuda", enabled=False):
            alpha = out["alpha"].float()
            yh = F.one_hot(y, K).float(); ys = yh * (1 - self.smooth) + self.smooth / K
            S = alpha.sum(1, keepdim=True); p = alpha / S
            mse = ((ys - p) ** 2 + p * (1 - p) / (S + 1)).sum(1).mean()
            lam = CFG["lambda_kl"] * min(1.0, epoch / CFG["kl_anneal_epochs"])
            return mse + lam * kl_dirichlet_to_uniform(yh + (1 - yh) * alpha)


class SoftmaxLoss(nn.Module):
    def __init__(self, smooth):
        super().__init__(); self.smooth = smooth

    def forward(self, out, y, epoch):
        return F.cross_entropy(out["logits"].float(), y, label_smoothing=self.smooth)


def make_criterion(variant):
    s = vcfg(variant, "label_smooth")
    return (EvidentialLoss if VARIANTS[variant]["head"] == "evidential" else SoftmaxLoss)(s)


# ══════════════════════════════════════════════════════════════════════════════
#  4. TRAIN / PREDICT
# ══════════════════════════════════════════════════════════════════════════════
class ModelEMA:
    def __init__(self, model, decay):
        self.decay, self.shadow = decay, copy.deepcopy(model).eval()
        for p in self.shadow.parameters(): p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        for se, e in zip(self.shadow.state_dict().values(), model.state_dict().values()):
            se.mul_(self.decay).add_(e.detach(), alpha=1 - self.decay) if se.dtype.is_floating_point else se.copy_(e)


@torch.no_grad()
def predict(model, loader, tta=None, with_fused=False):
    """Returns dict of arrays. TTA = horizontal flip: alpha (evidential) or probabilities (softmax) averaged."""
    tta = CFG["tta"] if tta is None else tta
    model.eval(); R = {k: [] for k in ["prob", "alpha", "vac", "label", "idx", "fused"]}
    for x, y, i in loader:
        x = x.to(DEVICE)
        with torch.autocast("cuda", enabled=False):                      # fp32 inference: no fp16 overflow -> NaN
            o = model(x.float()); o2 = model(torch.flip(x, [3]).float()) if tta else None
        if o["alpha"] is not None:
            alpha = o["alpha"] if o2 is None else (o["alpha"] + o2["alpha"]) / 2
            S = alpha.sum(1, keepdim=True); prob, vac = alpha / S, (K / S).squeeze(1)
        else:
            prob = o["prob"] if o2 is None else (o["prob"] + o2["prob"]) / 2
            alpha, vac = None, torch.full((x.size(0),), float("nan"), device=x.device)
        R["prob"].append(prob.float().cpu()); R["vac"].append(vac.float().cpu())
        R["label"].append(y); R["idx"].append(i)
        if alpha is not None: R["alpha"].append(alpha.float().cpu())
        if with_fused: R["fused"].append(o["fused"].float().cpu())
    out = {k: torch.cat(v).numpy() for k, v in R.items() if len(v)}
    return out


def preds_to_df(P, meta):
    """meta: DataFrame aligned with loader order. Saves the canonical prediction schema."""
    m = meta.iloc[P["idx"]].reset_index(drop=True)
    df = pd.DataFrame(P["prob"], columns=[f"p{k}" for k in range(K)])
    df.insert(0, "path", m.path.values); df.insert(1, "label", P["label"].astype(int))
    df.insert(2, "pred", P["prob"].argmax(1))
    if "alpha" in P:
        for k in range(K): df[f"alpha{k}"] = P["alpha"][:, k]
    df["vacuity"] = P["vac"]
    for c in ["cluster", "is_rep"]:
        if c in m: df[c] = m[c].values
    return df


def run_epoch(model, loader, criterion, opt, scaler, sched, epoch, ema, lr_log):
    model.train(); opt.zero_grad(set_to_none=True)
    tl = tc = tn = 0
    for step, (x, y, _) in enumerate(loader):
        x, y = x.to(DEVICE), y.to(DEVICE)
        with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
            out = model(x)
        loss = criterion(out, y, epoch) / CFG["grad_accum"]
        if not torch.isfinite(loss):                                     # NaN/inf guard: skip this batch, never backprop it
            nbad = globals().get("_NAN_BATCHES", 0) + 1; globals()["_NAN_BATCHES"] = nbad
            print(f"  [WARN] non-finite loss at step {step} (total {nbad}); batch skipped")
            if nbad > 50: raise RuntimeError("Too many non-finite losses: training diverged. Lower lr (e.g. 1e-4) or set amp off.")
            opt.zero_grad(set_to_none=True); continue
        scaler.scale(loss).backward()
        if (step + 1) % CFG["grad_accum"] == 0 or step + 1 == len(loader):
            scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), CFG["clip_grad"])
            scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad(set_to_none=True)
            ema.update(model); lr_log.append(opt.param_groups[0]["lr"])
        tl += loss.item() * CFG["grad_accum"] * y.size(0); tc += (out["prob"].argmax(1) == y).sum().item(); tn += y.size(0)
    return tl / tn, tc / tn


@torch.no_grad()
def eval_val(model, loader, criterion, epoch):
    """Validation loss/accuracy/NLL on the HELD-OUT validation split (never the test split)."""
    model.eval(); tl = tc = tn = nll = 0
    for x, y, _ in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        with torch.autocast("cuda", enabled=False):
            out = model(x)
        tl += criterion(out, y, epoch).item() * y.size(0)
        tc += (out["prob"].argmax(1) == y).sum().item(); tn += y.size(0)
        nll += -torch.log(out["prob"].gather(1, y[:, None]).clamp_min(1e-12)).sum().item()
    return tl / tn, tc / tn, nll / tn


class BudgetExceeded(Exception):
    pass


TRAIN_DEADLINE = None        # absolute time (s) after which no training may be running; set by worker_main()


def train_model(variant, seed, train_df, val_df, epochs, tag):
    set_seed(seed)
    model = WaveSSMTriNet(variant).to(DEVICE)
    crit = make_criterion(variant)
    tr_loader, va_loader = make_loader(train_df, True, seed), make_loader(val_df, False)
    opt = optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=CFG["lr"], weight_decay=CFG["weight_decay"])
    steps = math.ceil(len(tr_loader) / CFG["grad_accum"]) * epochs + 1
    sched = optim.lr_scheduler.OneCycleLR(opt, max_lr=CFG["lr"], total_steps=steps, pct_start=CFG["pct_start"],
                                          anneal_strategy="cos", div_factor=CFG["div_factor"], final_div_factor=CFG["final_div_factor"])
    scaler, ema = torch.amp.GradScaler("cuda", enabled=USE_AMP), ModelEMA(model, CFG["ema_decay"])
    hist, lr_log, ep_times, best = [], [], [], dict(key=(-1, 0), state=None, src="", ep=0)
    for ep in range(1, epochs + 1):
        t0 = time.time()
        tl, ta = run_epoch(model, tr_loader, crit, opt, scaler, sched, ep, ema, lr_log)
        (vl, va, vn), (_, ea, en) = eval_val(model, va_loader, crit, ep), eval_val(ema.shadow, va_loader, crit, ep)
        for src, acc, nll, net in (("raw", va, vn, model), ("ema", ea, en, ema.shadow)):
            key = (acc, -nll)                                            # select on VAL accuracy, tie-break VAL NLL
            if not math.isfinite(nll):
                print(f"  [WARN] {src} val NLL is non-finite at epoch {ep}; not eligible for selection"); continue
            if key > best["key"]:
                best.update(key=key, state=copy.deepcopy(net.state_dict()), src=src, ep=ep)
        hist.append(dict(epoch=ep, train_loss=tl, train_acc=ta, val_loss=vl, val_acc=va, ema_val_acc=ea, lr=opt.param_groups[0]["lr"]))
        print(f"  [{tag}] ep {ep:3d} tr {tl:.4f}/{ta*100:5.2f}  val {vl:.4f}/{va*100:5.2f}  ema {ea*100:5.2f}  {time.time()-t0:.0f}s", flush=True)
        ep_times.append(time.time() - t0)
        if ep == 1: print(f"  [{tag}] projected run time ~ {ep_times[0]*epochs/60:.0f} min for {epochs} epochs", flush=True)
        if TRAIN_DEADLINE and ep < epochs and time.time() + np.mean(ep_times[-3:]) * (epochs - ep) > TRAIN_DEADLINE + 600:
            raise BudgetExceeded(f"{tag}: projected end is past the time budget -> run aborted at epoch {ep}/{epochs}")
    if best["state"] is None: raise RuntimeError("No finite checkpoint was produced (all epochs gave NaN/inf). Try USE_AMP=False / lower lr.")
    print(f"  [{tag}] selected epoch {best['ep']} ({best['src']}) val_acc={best['key'][0]*100:.2f}")
    model.load_state_dict(best["state"])
    ck = os.path.join(CFG["out_dir"], f"ckpt_{tag}.pt"); torch.save(best["state"], ck)
    pd.DataFrame(hist).to_csv(os.path.join(CFG["out_dir"], f"history_{tag}.csv"), index=False)
    np.save(os.path.join(CFG["out_dir"], f"lr_steps_{tag}.npy"), np.array(lr_log))
    return model, dict(selected_epoch=best["ep"], selected_src=best["src"], val_acc=best["key"][0],
                       pretrained=getattr(getattr(model, "cnn_branch", None), "pretrained_loaded", None))


def eval_all_sets(model, manifest, tag, variant):
    """Scores val / test / external ONCE with the selected checkpoint, saves prediction CSVs."""
    res = {}
    out = CFG["out_dir"]
    for name, role in (("val", "val"), ("test", "test"), ("external", "external")):
        meta = manifest[manifest.role == role].reset_index(drop=True)
        if not len(meta): continue
        P = predict(model, make_loader(meta, False))
        df = preds_to_df(P, meta); sanity_check_pred_df(df, K, name=f"{tag}/{name}")
        df.to_csv(os.path.join(out, f"pred_{tag}_{name}.csv"), index=False)
        res[name] = df
    return res


# ══════════════════════════════════════════════════════════════════════════════
#  5. EXPERIMENT DRIVER + CROSS-SEED STATISTICS
# ══════════════════════════════════════════════════════════════════════════════
def main_job(manifest, variant, seed):
    """Train (or resume) one (variant, seed), score val/test/external once, write predictions + metric JSONs."""
    tr, va = manifest[manifest.role == "train"], manifest[manifest.role == "val"]
    tag = f"{variant}_s{seed}"
    print(f"\n{'='*70}\n  {tag}\n{'='*70}", flush=True)
    ck = os.path.join(CFG["out_dir"], f"ckpt_{tag}.pt")
    resumed = False
    need = [f"pred_{tag}_val.csv", f"pred_{tag}_test.csv", f"eval_{tag}_test_summary.json"]
    if CFG["resume"] and all(os.path.exists(os.path.join(CFG["out_dir"], f)) for f in need):
        print(f"  [RESUME] {tag}: predictions and evaluation already on disk -> nothing to do", flush=True); return True
    if CFG["resume"] and os.path.exists(ck) and os.path.exists(os.path.join(CFG["out_dir"], f"pred_{tag}_test.csv")):
        print(f"  [RESUME] {tag}: checkpoint found, skipping training", flush=True)
        model = WaveSSMTriNet(variant).to(DEVICE); model.load_state_dict(torch.load(ck, map_location=DEVICE)); model.eval(); resumed = True
    else:
        model, _ = train_model(variant, seed, tr, va, CFG["epochs"], tag)
    dfs = eval_all_sets(model, manifest, tag, variant)
    for name in ("test", "external"):
        if name not in dfs: continue
        d = dfs[name]; first = (seed == CFG["seeds"][0])
        evaluate_predictions(d, os.path.join(CFG["out_dir"], f"eval_{tag}_{name}"), val_df=dfs["val"], bootstrap=first, make_figs=first)
        if name == "test" and "is_rep" in d:
            evaluate_predictions(d, os.path.join(CFG["out_dir"], f"eval_{tag}_test_dedup"), val_df=dfs["val"],
                                 subset_mask=d.is_rep.values.astype(bool), bootstrap=False, make_figs=False)
    del model; torch.cuda.empty_cache()
    return resumed


def load_runs():
    """Rebuild the per-run summaries from files on disk (workers are separate processes)."""
    runs, out = {}, CFG["out_dir"]
    for variant in _variants_on_disk():
        for seed in seeds_for(variant):
            tag = f"{variant}_s{seed}"
            summ, dfs = {}, {}
            for name in ("test", "test_dedup", "external"):
                p = os.path.join(out, f"eval_{tag}_{name}_summary.json")
                if os.path.exists(p): summ[name] = json.load(open(p))
            for name in ("val", "test", "external"):
                p = os.path.join(out, f"pred_{tag}_{name}.csv")
                if os.path.exists(p): dfs[name] = pd.read_csv(p)
            if "test" in summ and "test" in dfs:
                runs[(variant, seed)] = dict(summary=summ, dfs=dfs)
    missing = [f"{v}_s{s}" for v in CFG["variants"] for s in seeds_for(v) if (v, s) not in runs]
    print(f"[RUNS] completed {len(runs)} (requested {sum(len(seeds_for(v)) for v in CFG['variants'])})" + (f"   NOT completed: {missing}" if missing else ""))
    return runs


def aggregate(runs):
    rows, tests = [], []
    metrics = ["accuracy", "macro_f1", "mcc", "kappa", "mean_auc"]
    variants = sorted({v for v, _ in runs}, key=list(VARIANTS).index)
    seeds_of = {v: [s for s in seeds_for(v) if (v, s) in runs] for v in variants}
    for variant in variants:
        for setname in ("test", "test_dedup", "external"):
            have = [s for s in seeds_of[variant] if setname in runs[(variant, s)]["summary"]]
            if not have: continue
            vals = {m: [runs[(variant, s)]["summary"][setname]["metrics"][m] for s in have] for m in metrics}
            row = dict(variant=variant, set=setname, n_seeds=len(have), seeds=",".join(map(str, have)), protocol=r2_protocol(have))
            for m in metrics:
                st = summarize_seeds(vals[m]); row[f"{m}_mean"], row[f"{m}_sd"] = st["mean"], st["sd"]
                row[f"{m}_ci95_lo"], row[f"{m}_ci95_hi"] = st["ci95"]
            for cal in ("ece", "brier", "nll"):
                c = [runs[(variant, s)]["summary"][setname]["calibration_raw"][cal] for s in have]
                row[f"{cal}_mean"] = float(np.mean(c)); row[f"{cal}_sd"] = float(np.std(c, ddof=1)) if len(c) > 1 else float("nan")
            rows.append(row)
    if not rows:
        print("[AGG] no completed runs -> nothing to aggregate"); return
    pd.DataFrame(rows).to_csv(os.path.join(CFG["out_dir"], "table_multiseed_summary.csv"), index=False)
    # paired comparisons vs the primary variant, on the seeds both variants completed
    base = CFG["primary_variant"]
    for variant in variants:
        if variant == base: continue
        common = [s for s in seeds_for(base) if (base, s) in runs and (variant, s) in runs]
        for setname in ("test", "external"):
            ok = [s for s in common if setname in runs[(base, s)]["dfs"] and setname in runs[(variant, s)]["dfs"]]
            if len(ok) < 2: continue
            a = [runs[(base, s)]["summary"][setname]["metrics"]["accuracy"] for s in ok]
            b = [runs[(variant, s)]["summary"][setname]["metrics"]["accuracy"] for s in ok]
            t = paired_seed_tests(a, b); t.update(variant=variant, vs=base, set=setname, seeds=ok)
            t["mcnemar_p_per_seed"] = [mcnemar_exact((runs[(base, s)]["dfs"][setname].pred == runs[(base, s)]["dfs"][setname].label).values,
                                                        (runs[(variant, s)]["dfs"][setname].pred == runs[(variant, s)]["dfs"][setname].label).values)["p"]
                                       for s in ok]
            tests.append(t)
    json.dump(tests, open(os.path.join(CFG["out_dir"], "table_paired_tests.json"), "w"), indent=2)
    print(pd.DataFrame(rows)[["variant", "set", "n_seeds", "accuracy_mean", "accuracy_sd", "macro_f1_mean", "mean_auc_mean", "ece_mean"]])


# ══════════════════════════════════════════════════════════════════════════════
#  6. CROSS-VALIDATION  (group-aware, inner validation for checkpoint selection; comment 7)
# ══════════════════════════════════════════════════════════════════════════════
def cv_split(manifest, f):
    pool = manifest[manifest.origin.isin(["brisc_train", "brisc_test"])].reset_index(drop=True)
    skf = StratifiedGroupKFold(CFG["cv_folds"], shuffle=True, random_state=CFG["split_seed"])
    tr_i, te_i = list(skf.split(np.zeros(len(pool)), pool.label.values, pool.cluster.values))[f - 1]
    return pool, tr_i, te_i


def cv_job(manifest, f):
    """One group-aware CV fold with an inner validation split for checkpoint selection (never the held-out fold)."""
    pool, tr_i, te_i = cv_split(manifest, f)
    ep = CFG["cv_epochs"] or CFG["epochs"]
    tag = f"cv{f}"
    print(f"\n{'='*70}\n  {tag}\n{'='*70}", flush=True)
    pf = os.path.join(CFG["out_dir"], f"pred_{tag}_test.csv")
    if CFG["resume"] and os.path.exists(pf):
        print(f"  [RESUME] {tag}: predictions found, skipping training", flush=True); return True
    tr_all = pool.iloc[tr_i].reset_index(drop=True)
    a, b = group_stratified_split(tr_all.label.values, tr_all.cluster.values, CFG["val_frac"], CFG["split_seed"] + f)
    model, _ = train_model(CFG["primary_variant"], CFG["seeds"][0], tr_all.iloc[a], tr_all.iloc[b], ep, tag)
    te = pool.iloc[te_i].reset_index(drop=True)
    df = preds_to_df(predict(model, make_loader(te, False)), te); sanity_check_pred_df(df, K, name=tag)
    df.to_csv(pf, index=False); del model; torch.cuda.empty_cache()
    return False


def cv_aggregate():
    rows = []
    for f in range(1, CFG["cv_folds"] + 1):
        pf = os.path.join(CFG["out_dir"], f"pred_cv{f}_test.csv")
        if not os.path.exists(pf): continue
        df = pd.read_csv(pf)
        m = classification_metrics(df.label.values, df.pred.values, df[[f"p{k}" for k in range(K)]].values)
        rows.append(dict(fold=f, **{k: m[k] for k in ["accuracy", "macro_f1", "mcc", "kappa", "mean_auc"]},
                         **{f"auc_{c}": v["auc"] for c, v in m["per_class"].items()}))
    print(f"[CV] completed {len(rows)}/{CFG['cv_folds']} folds")
    if not rows: return None
    df = pd.DataFrame(rows); df.to_csv(os.path.join(CFG["out_dir"], "table_cv_folds.csv"), index=False)
    print(df.describe().loc[["mean", "std"]] if len(df) > 1 else df)
    return df


# ══════════════════════════════════════════════════════════════════════════════
#  7. ANALYSES: occlusion/Shapley, Grad-CAM, t-SNE, complexity, domain shift
# ══════════════════════════════════════════════════════════════════════════════
def load_primary(variant, seed):
    m = WaveSSMTriNet(variant).to(DEVICE)
    m.load_state_dict(torch.load(os.path.join(CFG["out_dir"], f"ckpt_{variant}_s{seed}.pt"), map_location=DEVICE))
    return m.eval()


@torch.no_grad()
def set_branch_means(model, loader, max_batches=40):
    sums = {n: 0.0 for n in model.branches}; cnt = 0
    for b, (x, _, _) in enumerate(loader):
        if b >= max_batches: break
        toks, _ = model.branch_tokens(x.to(DEVICE))
        for n in sums: sums[n] = sums[n] + toks[n].float().sum(0, keepdim=True)
        cnt += x.size(0)
    for n in sums: getattr(model, f"mean_{n}").copy_(sums[n] / cnt)


def occlusion_and_shapley(model, train_df, test_df):
    set_branch_means(model, make_loader(train_df.sample(min(1024, len(train_df)), random_state=0), False))
    loader = make_loader(test_df, False); names = list(model.branches)
    vals = {}
    for r in range(len(names) + 1):
        for present in combinations(names, r):
            drop = tuple(n for n in names if n not in present)
            c = n_ = 0
            with torch.no_grad():
                for x, y, _ in loader:
                    with torch.autocast("cuda", enabled=False):
                        o = model(x.to(DEVICE), drop=drop)
                    c += (o["prob"].argmax(1).cpu() == y).sum().item(); n_ += y.size(0)
            vals[frozenset(present)] = c / n_
    phi = shapley_values(vals, names)
    full = vals[frozenset(names)]
    out = dict(full_accuracy=full, coalitions={"+".join(sorted(k)) or "none": v for k, v in vals.items()},
               single_branch_removal_drop={n: full - vals[frozenset(set(names) - {n})] for n in names}, shapley_accuracy=phi)
    json.dump(out, open(os.path.join(CFG["out_dir"], "analysis_branch_occlusion_shapley.json"), "w"), indent=2)
    fig, ax = plt.subplots(figsize=(5, 4)); ax.bar([R2_BRANCH_LABEL.get(k, k) for k in phi], [100 * v for v in phi.values()], color=PALETTE[:len(phi)])
    ax.set_ylabel("Shapley value of accuracy (pp)"); ax.set_title("Branch contribution (occlusion/Shapley)", fontweight="bold")
    fig.tight_layout(); fig.savefig(os.path.join(CFG["out_dir"], "fig_branch_shapley.png"), dpi=300); plt.close(fig)
    return out


def gradcam_all_branches(model, test_df, n_per_class=2):
    """Grad-CAM for each branch's feature map w.r.t. the predicted-class logit of the FULL model."""
    model.eval(); model.capture = True
    mean, std = np.array([0.485, 0.456, 0.406]), np.array([0.229, 0.224, 0.225])
    hits, rows = {n: [] for n in model.branches}, []
    pick = pd.concat([g.sample(min(n_per_class, len(g)), random_state=0) for _, g in test_df.groupby("label")])
    for _, r in pick.iterrows():
        with Image.open(r.path) as im:
            x = build_transforms(False)(im.convert("RGB")).unsqueeze(0).to(DEVICE).requires_grad_(True)
        out = model(x); c = out["logits"].argmax(1).item()
        model.zero_grad(); out["logits"][0, c].backward()
        cams = {}
        for n, m in model._maps.items():
            if m.grad is None: continue
            w = m.grad.mean((2, 3), keepdim=True); cam = F.relu((w * m).sum(1, keepdim=True))
            cam = F.interpolate(cam, size=(CFG["img_size"],) * 2, mode="bilinear", align_corners=False)[0, 0].detach().cpu().numpy()
            cams[n] = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        img = (x[0].detach().cpu().permute(1, 2, 0).numpy() * std + mean).clip(0, 1)
        rows.append((img, cams, int(r.label), c))
        if CFG["mask_root"]:                                            # optional pointing-game hit rate
            stem = os.path.splitext(os.path.basename(r.path))[0]
            for ext in (".png", ".jpg", ".tif"):
                mp = os.path.join(CFG["mask_root"], stem + ext)
                if os.path.exists(mp):
                    mk = np.asarray(Image.open(mp).convert("L").resize((CFG["img_size"],) * 2)) > 0
                    for n, cam in cams.items(): hits[n].append(bool(mk[np.unravel_index(cam.argmax(), cam.shape)]))
                    break
    model.capture = False
    names = list(model.branches)
    fig, axes = plt.subplots(len(rows), 1 + len(names), figsize=(2.6 * (1 + len(names)), 2.6 * len(rows)))
    axes = np.atleast_2d(axes)
    for i, (img, cams, yt, yp) in enumerate(rows):
        axes[i, 0].imshow(img); axes[i, 0].set_title(f"true={CFG['class_names'][yt]}\npred={CFG['class_names'][yp]}", fontsize=8)
        for j, n in enumerate(names, 1):
            axes[i, j].imshow(img); axes[i, j].imshow(cams.get(n, np.zeros_like(img[..., 0])), cmap="jet", alpha=0.45)
            if i == 0: axes[i, j].set_title(f"Grad-CAM: {R2_BRANCH_LABEL.get(n, n)}", fontsize=8)
    for a in axes.ravel(): a.axis("off")
    fig.suptitle(f"Grad-CAM of each branch w.r.t. the full-model decision (qualitative; not a localisation claim; seed {CFG['seeds'][0]}, official test images)", fontsize=9)
    fig.tight_layout(); fig.savefig(os.path.join(CFG["out_dir"], "fig_gradcam_all_branches.png"), dpi=300); plt.close(fig)
    if CFG["mask_root"]:
        json.dump({n: (float(np.mean(h)) if h else None) for n, h in hits.items()}, open(os.path.join(CFG["out_dir"], "analysis_gradcam_hit_rate.json"), "w"), indent=2)


# ══════════════════════════════════════════════════════════════════════════════
#  R2 helpers that need the training environment (resume, CV manifest, Grad-CAM statistics, perturbations)
# ══════════════════════════════════════════════════════════════════════════════
def r2_restore_previous_outputs():
    """Copy checkpoints / predictions / evaluation files of an earlier Kaggle session into out_dir (never overwrites)."""
    import fnmatch, shutil
    src = CFG.get("resume_from")
    if not src or not os.path.isdir(src) or os.path.abspath(src) == os.path.abspath(CFG["out_dir"]): return
    pats = ("ckpt_*.pt", "pred_*.csv", "eval_*", "history_*.csv", "lr_steps_*.npy", "brisc_thumbs.npy", "ext_thumbs.npy", "table_cv_folds.csv",
            "split_manifest.csv", "run_config.json", "variants.json", "analysis_*.json", "duplicate_report.json")
    n = 0
    for root, _, files in os.walk(src):
        for fn in files:
            if any(fnmatch.fnmatch(fn, p) for p in pats):
                dst = os.path.join(CFG["out_dir"], fn)
                if not os.path.exists(dst): shutil.copy2(os.path.join(root, fn), dst); n += 1
    print(f"[RESUME] restored {n} files from {src}")


def r2_verify_resume_manifest(manifest):
    """Runs from different sessions are only comparable if the split is identical -> fail loudly if it changed."""
    src = CFG.get("resume_from")
    p = os.path.join(src, "split_manifest.csv") if src else None
    if not p or not os.path.exists(p) or os.path.abspath(src) == os.path.abspath(CFG["out_dir"]): return
    old = pd.read_csv(p); m = old[["path", "role", "cluster"]].merge(manifest[["path", "role", "cluster"]], on="path", suffixes=("_old", "_new"))
    bad = int(((m.role_old != m.role_new) | (m.cluster_old != m.cluster_new)).sum())
    if bad: raise RuntimeError(f"split changed since the previous session ({bad} images differ in role/cluster): results would not be comparable")
    print(f"[RESUME] split manifest identical to the previous session on {len(m)} images")


def dump_cv_manifest(manifest):
    rows = []
    for f in range(1, CFG["cv_folds"] + 1):
        pool, _, te_i = cv_split(manifest, f)
        rows.append(pd.DataFrame(dict(path=pool.path.values[te_i], fold=f, label=pool.label.values[te_i], cluster=pool.cluster.values[te_i])))
    pd.concat(rows).to_csv(os.path.join(CFG["out_dir"], "cv_folds_manifest.csv"), index=False)


def _r2_safe(name, fn, *a, **k):
    try: return fn(*a, **k)
    except Exception: print(f"[R2] {name} failed (continuing):\n{traceback.format_exc()}", flush=True)


def r2_cams_for(model, path):
    """Branch Grad-CAMs (img-size, [0,1]) w.r.t. the predicted-class logit of the FULL model. Returns img, cams, predicted class, confidence."""
    mean, std = np.array([0.485, 0.456, 0.406]), np.array([0.229, 0.224, 0.225])
    with Image.open(path) as im:
        x = build_transforms(False)(im.convert("RGB")).unsqueeze(0).to(DEVICE).requires_grad_(True)
    out = model(x); c = out["logits"].argmax(1).item(); conf = float(out["prob"][0, c].detach())
    model.zero_grad(); out["logits"][0, c].backward()
    cams = {}
    for n, m in model._maps.items():
        if m.grad is None: continue
        w = m.grad.mean((2, 3), keepdim=True); cam = F.relu((w * m).sum(1, keepdim=True))
        cam = F.interpolate(cam, size=(CFG["img_size"],) * 2, mode="bilinear", align_corners=False)[0, 0].detach().cpu().numpy()
        cams[n] = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    img = (x[0].detach().cpu().permute(1, 2, 0).numpy() * std + mean).clip(0, 1)
    return img, cams, c, conf


def r2_load_lesion_mask(path, size):
    if not CFG["mask_root"]: return None
    stem = os.path.splitext(os.path.basename(path))[0]
    for ext in (".png", ".jpg", ".tif"):
        mp = os.path.join(CFG["mask_root"], stem + ext)
        if os.path.exists(mp): return np.asarray(Image.open(mp).convert("L").resize((size, size), Image.NEAREST)) > 0
    return None


def r2_gradcam_failures(model, pred_df, n=8):
    """R2-8: same three-branch Grad-CAM layout for MISCLASSIFIED test images (most frequent confusions first)."""
    pick = r2_pick_failures(pred_df, n)
    if pick.empty: print("[R2] no misclassified test images"); return
    model.eval(); model.capture = True; rows = []
    for _, r in pick.iterrows():
        img, cams, c, conf = r2_cams_for(model, r.path); rows.append((img, cams, int(r.label), c, conf))
    model.capture = False
    names = list(model.branches); fig, axes = plt.subplots(len(rows), 1 + len(names), figsize=(2.6 * (1 + len(names)), 2.6 * len(rows))); axes = np.atleast_2d(axes)
    for i, (img, cams, yt, yp, conf) in enumerate(rows):
        axes[i, 0].imshow(img); axes[i, 0].set_title(f"true={CFG['class_names'][yt]}\npred={CFG['class_names'][yp]} (p={conf:.2f})", fontsize=8)
        for j, n_ in enumerate(names, 1):
            axes[i, j].imshow(img); axes[i, j].imshow(cams.get(n_, np.zeros_like(img[..., 0])), cmap="jet", alpha=0.45)
            if i == 0: axes[i, j].set_title(f"Grad-CAM: {R2_BRANCH_LABEL.get(n_, n_)}", fontsize=8)
    for a in axes.ravel(): a.axis("off")
    fig.suptitle(f"Misclassified official-test images (seed {CFG['seeds'][0]}); Grad-CAM w.r.t. the predicted class of the full model (qualitative)", fontsize=9)
    fig.tight_layout(); fig.savefig(os.path.join(CFG["out_dir"], "fig_gradcam_failure_cases.png"), dpi=300); plt.close(fig)
    pick[["path", "label", "pred"]].to_csv(os.path.join(CFG["out_dir"], "analysis_r2_failure_cases.csv"), index=False)


def r2_gradcam_quant(model, test_df, per_class=100):
    """R2-8: where do the maps put their mass? fraction outside the head, in the four image corners, peak inside the head, and (with
    CFG['mask_root']) pointing-game hit / energy in the lesion. Stratified sample of test images, per branch, correct vs misclassified."""
    model.eval(); model.capture = True; rows = []
    pick = pd.concat([g.sample(min(per_class, len(g)), random_state=0) for _, g in test_df.groupby("label")])
    for _, r in pick.iterrows():
        img, cams, c, _ = r2_cams_for(model, r.path); hm = r2_head_mask(img.mean(2)); lesion = r2_load_lesion_mask(r.path, CFG["img_size"])
        for n, cam in cams.items():
            rows.append(dict(branch=n, path=r.path, label=int(r.label), pred=c, correct=bool(c == int(r.label)), **r2_cam_stats(cam, hm, lesion)))
    model.capture = False
    df = pd.DataFrame(rows); df.to_csv(os.path.join(CFG["out_dir"], "analysis_r2_gradcam_stats.csv"), index=False)
    summ = r2_cam_summary(rows); summ.to_csv(os.path.join(CFG["out_dir"], "analysis_r2_gradcam_summary.csv"), index=False)
    print("[R2] Grad-CAM mass statistics (fraction outside head / in corners; lesion columns need mask_root):\n", summ.round(3).to_string())
    return summ


class R2PerturbDataset(Dataset):
    """Test images with one test-time perturbation applied after the standard resize (clean = same pipeline, no change)."""
    def __init__(self, df, kind):
        self.paths, self.y, self.kind = df.path.tolist(), df.label.tolist(), kind
        self.norm = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])

    def __len__(self): return len(self.paths)

    def __getitem__(self, i):
        S = CFG["img_size"]
        with Image.open(self.paths[i]) as im:
            a = np.asarray(im.convert("RGB").resize((S, S), Image.BILINEAR), np.float32) / 255.0
        a = r2_perturb_array(a, self.kind, r2_head_mask(a.mean(2)), seed=i)
        return self.norm(torch.from_numpy(np.ascontiguousarray(a.transpose(2, 0, 1))).float()), self.y[i], i


def r2_perturbation_eval(model, test_df, bs=64):
    """R2-8: accuracy under brightness / contrast / gamma / noise / background removal / corner masking (reliance on acquisition cues)."""
    preds = {}
    for kind in R2_PERTURBATIONS:
        dl = DataLoader(R2PerturbDataset(test_df, kind), bs, shuffle=False, num_workers=CFG["num_workers"])
        preds[kind] = predict(model, dl, tta=False)["prob"].argmax(1)
    tab = r2_perturbation_table(test_df.label.values, preds, test_df["cluster"].values if "cluster" in test_df else None)
    tab.to_csv(os.path.join(CFG["out_dir"], "analysis_r2_perturbation.csv"), index=False)
    print("[R2] test-time perturbations (accuracy, change vs clean):\n", tab.round(4).to_string(index=False))
    return tab


def embeddings(model, test_df):
    from sklearn.manifold import TSNE
    sub = pd.concat([g.sample(min(CFG["tsne_per_class"], len(g)), random_state=0) for _, g in test_df.groupby("label")]).reset_index(drop=True)
    P = predict(model, make_loader(sub, False), tta=False, with_fused=True)
    Z, y = P["fused"], P["label"]
    perp = min(30, max(5, len(y) // 4))
    ts = TSNE(2, perplexity=perp, init="pca", random_state=0).fit_transform(Z)   # sklearn default: 1000 iterations
    fig_embedding(ts, y, CFG["class_names"], os.path.join(CFG["out_dir"], "fig_tsne_balanced.png"),
                     f"t-SNE of fused features (stratified {CFG['tsne_per_class']}/class, perplexity {perp}, PCA init, seed 0)")
    if "plane" in sub and sub["plane"].nunique() > 1:                                   # R2-8: does the embedding encode acquisition plane?
        r2_fig_scatter_by(ts, sub["plane"].values, os.path.join(CFG["out_dir"], "fig_tsne_by_plane.png"),
                          "t-SNE of fused features coloured by imaging plane (same embedding as fig_tsne_balanced)")
        probe = dict(class_from_embedding=r2_linear_probe(Z, y), plane_from_embedding=r2_linear_probe(Z, sub["plane"].values),
                     n=int(len(y)), planes={str(k): int(v) for k, v in sub["plane"].value_counts().items()})
        json.dump(probe, open(os.path.join(CFG["out_dir"], "analysis_r2_embedding_probe.json"), "w"), indent=2)
        print("[R2] linear probes on the fused embedding:", json.dumps(probe)[:400])
    try:
        import umap
        um = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=0).fit_transform(Z)
        fig_embedding(um, y, CFG["class_names"], os.path.join(CFG["out_dir"], "fig_umap_balanced.png"), "UMAP of fused features (n_neighbors=15, min_dist=0.1)")
    except Exception as e:
        print("[UMAP] skipped:", e)


def scan_flops_estimate(L=196, D=192, S=16, layers=3):
    """Analytic cost of the custom scan + discretisation (elementwise ops are NOT counted by FlopCounterMode)."""
    steps = math.ceil(math.log2(L))
    per = L * D * S * (3 * steps + 6)        # 3 mul/add per scan step; exp, dB x, C h, reduction
    return layers * per


def complexity_and_speed(model):
    from torch.utils.flop_counter import FlopCounterMode
    model.eval(); x = torch.randn(1, 3, CFG["img_size"], CFG["img_size"], device=DEVICE)
    rep = {}
    with torch.no_grad():
        with FlopCounterMode(display=False) as fc: model(x)
        rep["total_flops_G_matmul_conv"] = fc.get_total_flops() / 1e9
        toks, _ = model.branch_tokens(x)
        for n, mod in (("wt", "wavelet_branch"), ("ssm", "ssm_branch"), ("cnn", "cnn_branch")):
            if n in model.branches:
                with FlopCounterMode(display=False) as f2: getattr(model, mod)(x)
                rep[f"{n}_flops_G"] = f2.get_total_flops() / 1e9
        with FlopCounterMode(display=False) as f3: z = model.fusion(toks)
        rep["fusion_flops_G"] = f3.get_total_flops() / 1e9
        with FlopCounterMode(display=False) as f4: model.head(z)
        rep["head_flops_G"] = f4.get_total_flops() / 1e9
    rep["ssm_scan_elementwise_flops_G_estimate"] = scan_flops_estimate() / 1e9 if "ssm" in model.branches else 0.0
    rep["total_flops_G_incl_scan"] = rep["total_flops_G_matmul_conv"] + rep["ssm_scan_elementwise_flops_G_estimate"]     # the ONE GFLOPs figure to report
    rep["total_params_M"] = sum(p.numel() for p in model.parameters()) / 1e6
    rep["trainable_params_M"] = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
    rows = [(n, p.numel(), p.requires_grad) for n, p in model.named_parameters()]       # layer-wise table (comment 14)
    pd.DataFrame(rows, columns=["parameter", "numel", "trainable"]).to_csv(os.path.join(CFG["out_dir"], "table_layerwise_params.csv"), index=False)
    rep["module_params_M"] = {n: sum(p.numel() for p in m.parameters()) / 1e6 for n, m in model.named_children()}
    speed = {}
    for bs in (1, 32):
        xb = torch.randn(bs, 3, CFG["img_size"], CFG["img_size"], device=DEVICE)
        for amp in ([False, True] if USE_AMP else [False]):
            with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16, enabled=amp):
                for _ in range(2 if SMOKE else 20): model(xb)
                if USE_AMP: torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
                ts = []
                for _ in range(3 if SMOKE else 100):
                    t0 = time.perf_counter(); model(xb)
                    if USE_AMP: torch.cuda.synchronize()
                    ts.append((time.perf_counter() - t0) * 1000)
            speed[f"bs{bs}_{'fp16' if amp else 'fp32'}"] = dict(
                latency_ms_mean=float(np.mean(ts)), latency_ms_sd=float(np.std(ts)), throughput_img_s=float(bs / (np.mean(ts) / 1000)),
                peak_mem_MB=float(torch.cuda.max_memory_allocated() / 2 ** 20) if USE_AMP else None)
    rep["speed"] = speed; rep["device"] = torch.cuda.get_device_name(0) if USE_AMP else "cpu"
    json.dump(rep, open(os.path.join(CFG["out_dir"], "analysis_complexity.json"), "w"), indent=2)
    return rep


def attention_entropy(model, loader, max_batches=10):
    """Evidence that TCAFG attention is non-degenerate (comment 12): entropy / log(N_k), 1.0 = uniform."""
    if not isinstance(model.fusion, TCAFG): return None
    ent = {}
    with torch.no_grad():
        for b, (x, _, _) in enumerate(loader):
            if b >= max_batches: break
            o = model(x.to(DEVICE), return_attn=True)
            for k, w in o["attn"].items():
                w = w.float().clamp_min(1e-12)
                ent.setdefault(k, []).append((-(w * w.log()).sum(-1) / math.log(w.size(-1))).mean().item())
    out = {k: float(np.mean(v)) for k, v in ent.items()}
    json.dump(out, open(os.path.join(CFG["out_dir"], "analysis_attention_entropy.json"), "w"), indent=2)
    return out


def domain_shift_table(manifest):
    """Acquisition-shift descriptors BRISC-test vs external, per class (comments 25, 28)."""
    a = manifest[manifest.role == "test"].sample(frac=1, random_state=0).groupby("label").head(300)
    b = manifest[manifest.role == "external"].sample(frac=1, random_state=0).groupby("label").head(300) if (manifest.role == "external").any() else None
    parts = [image_intensity_stats(a.path.tolist()).assign(set="brisc_test", label=a.label.values)]
    if b is not None: parts.append(image_intensity_stats(b.path.tolist()).assign(set="external", label=b.label.values))
    d = pd.concat(parts); d["class"] = d.label.map(dict(enumerate(CFG["class_names"])))
    t = d.groupby(["set", "class"])[["mean", "std", "bg_frac", "fg_mean", "p95"]].agg(["mean", "std"]); t.to_csv(os.path.join(CFG["out_dir"], "analysis_domain_shift_intensity.csv"))
    print(t)


# ══════════════════════════════════════════════════════════════════════════════
#  8. SMOKE-TEST DATA, REPRODUCIBILITY DUMP, MAIN
# ══════════════════════════════════════════════════════════════════════════════
def make_smoke_data():
    rng = np.random.RandomState(0)
    for root, splits, n in (("/tmp/smoke/brisc/classification_task", {"train": 10, "test": 4}, None), ("/tmp/smoke/ext/Training", None, 5)):
        for sp, cnt in (splits.items() if splits else [("", n)]):
            for c in CFG["class_names"]:
                d = os.path.join(root, sp, c) if sp else os.path.join(root, c); os.makedirs(d, exist_ok=True)
                for i in range(cnt):
                    a = (rng.rand(64, 64, 3) * 120 + 40 * CFG["class_names"].index(c)).clip(0, 255).astype(np.uint8)
                    Image.fromarray(a).save(os.path.join(d, f"{c}_{i}.jpg"))
    shutil_src = "/tmp/smoke/brisc/classification_task/test/glioma/glioma_0.jpg"       # planted train/test duplicate
    Image.open(shutil_src).save("/tmp/smoke/brisc/classification_task/train/glioma/dup_of_test.jpg")


def dump_reproducibility():
    json.dump({k: v for k, v in CFG.items()}, open(os.path.join(CFG["out_dir"], "run_config.json"), "w"), indent=2, default=str)
    json.dump(VARIANTS, open(os.path.join(CFG["out_dir"], "variants.json"), "w"), indent=2)
    with open(os.path.join(CFG["out_dir"], "environment.txt"), "w") as f:
        f.write(f"python {sys.version}\ntorch {torch.__version__}\ntimm {timm.__version__}\nnumpy {np.__version__}\npandas {pd.__version__}\n"
                f"device {DEVICE}\ngpus {[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())] if torch.cuda.is_available() else []}\n"
                f"cudnn_deterministic {torch.backends.cudnn.deterministic}\ncudnn_benchmark {torch.backends.cudnn.benchmark}\n"
                f"RUN_MODE {RUN_MODE}  R2_STAGE {R2_STAGE}\n")


# ══════════════════════════════════════════════════════════════════════════════
#  9. MULTI-GPU ORCHESTRATION (one worker process per GPU) + TIME BUDGET
# ══════════════════════════════════════════════════════════════════════════════
def job_list():
    """Priority: seeds 42-44 of every requested variant -> seeds 45-46 of the five-seed variants (the heads comparison must be complete)
    -> CV folds. Finished jobs are skipped instantly; a time-budget cut drops the tail."""
    seeds, variants = CFG["seeds"], CFG["variants"]
    jobs = [("main", v, s) for s in seeds[:3] for v in variants if s in seeds_for(v)]
    jobs += [("main", v, s) for s in seeds[3:] for v in variants if s in seeds_for(v)]
    if CFG["run_cv"]: jobs += [("cv", f) for f in range(1, CFG["cv_folds"] + 1)]
    return jobs


def job_name(j):
    return f"main_{j[1]}_s{j[2]}" if j[0] == "main" else f"cv_{j[1]}"


def claim(job):
    """Atomic cross-process job claim (O_EXCL file) -> dynamic load balancing between the GPU workers."""
    d = os.path.join(CFG["out_dir"], "claims"); os.makedirs(d, exist_ok=True)
    try:
        os.close(os.open(os.path.join(d, job_name(job)), os.O_CREAT | os.O_EXCL | os.O_WRONLY)); return True
    except FileExistsError:
        return False


def worker_main(manifest=None):
    global TRAIN_DEADLINE
    rank, world = int(os.environ.get("WAVESSM_WORKER", "0")), int(os.environ.get("WAVESSM_WORLD", "1"))
    TRAIN_DEADLINE = T0 + CFG["time_budget_h"] * 3600 - CFG["analysis_reserve_min"] * 60
    if not SMOKE: CFG["num_workers"] = max(2, (os.cpu_count() or 4) // world)
    if manifest is None: manifest = pd.read_csv(os.path.join(CFG["out_dir"], "split_manifest.csv"))
    print(f"[WORKER {rank}/{world}] device={DEVICE} " + (torch.cuda.get_device_name(0) if USE_AMP else "") +
          f"  loader_workers={CFG['num_workers']}  train deadline in {(TRAIN_DEADLINE - time.time()) / 3600:.2f} h", flush=True)
    dur = {}
    for job in job_list():
        if not claim(job): continue
        kind = job[0]
        est = dur.get("main") if kind == "main" else dur.get("cv", dur["main"] * (CFG["cv_epochs"] or CFG["epochs"]) / CFG["epochs"] if "main" in dur else None)
        if est and time.time() + 1.10 * est > TRAIN_DEADLINE:
            print(f"[BUDGET] worker {rank}: skipping {job_name(job)} (needs ~{est / 60:.0f} min, "
                  f"{max(0, TRAIN_DEADLINE - time.time()) / 60:.0f} min left for training)", flush=True); continue
        t = time.time()
        try:
            resumed = main_job(manifest, job[1], job[2]) if kind == "main" else cv_job(manifest, job[1])
            if not resumed: dur[kind] = time.time() - t
            print(f"[JOB DONE] {job_name(job)}  {(time.time() - t) / 60:.1f} min  (total elapsed {(time.time() - T0) / 3600:.2f} h)", flush=True)
        except BudgetExceeded as e:
            print(f"[BUDGET] {e}", flush=True)
        except Exception:
            print(f"[JOB FAILED] {job_name(job)}\n{traceback.format_exc()}", flush=True)
        if DEVICE.type == "cuda": torch.cuda.empty_cache()
    print(f"[WORKER {rank}] finished", flush=True)


def _script_path():
    """Path of this script, or (inside a notebook) the current cell's source written to a file."""
    if "__file__" in globals(): return os.path.abspath(__file__)
    try:
        src = get_ipython().user_ns["In"][-1]                           # noqa: F821  (the cell being executed)
    except Exception:
        return None
    if src.lstrip().startswith("%%"): src = src.split("\n", 1)[1]
    if "def worker_main" not in src or "WAVESSM-TRINET-SCRIPT" not in src: return None
    p = os.path.join(CFG["out_dir"], "_wavessm_script.py")
    with open(p, "w", encoding="utf-8") as f: f.write(src)
    return p


def parallel_world():
    n = int(os.environ.get("WAVESSM_FORCE_WORKERS", "0")) or (torch.cuda.device_count() if torch.cuda.is_available() else 0)
    return n if n >= 2 else 1


def _monitor(procs, paths):
    pos = [0] * len(procs)
    def drain():
        for i, p in enumerate(paths):
            with open(p, "rb") as f:
                f.seek(pos[i]); chunk = f.read()
            k = chunk.rfind(b"\n")
            if k < 0: continue
            pos[i] += k + 1
            for line in chunk[:k + 1].decode("utf-8", "replace").splitlines():
                if line.strip(): print(f"[gpu{i}] {line}", flush=True)
    while any(p.poll() is None for p in procs):
        drain(); time.sleep(5)
    drain()
    return [p.returncode for p in procs]


def run_jobs(manifest):
    shutil_rmtree = __import__("shutil").rmtree
    shutil_rmtree(os.path.join(CFG["out_dir"], "claims"), ignore_errors=True)
    n = parallel_world()
    path = _script_path() if n >= 2 else None
    if n >= 2 and path is None:
        print("[PARALLEL] could not locate this script's source (run it as ONE notebook cell or as a .py file) -> using a single GPU")
        n = 1
    if n < 2:
        print("[PARALLEL] single process"); worker_main(manifest); return
    print(f"[PARALLEL] launching {n} worker processes (one per GPU); jobs in priority order: {[job_name(j) for j in job_list()]}")
    procs, logs, fhs = [], [], []
    try:
        for r in range(n):
            env = dict(os.environ, WAVESSM_WORKER=str(r), WAVESSM_WORLD=str(n), WAVESSM_T0=str(T0), RUN_MODE=RUN_MODE)
            if torch.cuda.is_available(): env["CUDA_VISIBLE_DEVICES"] = str(r)
            lp = os.path.join(CFG["out_dir"], f"worker{r}.log"); fh = open(lp, "w")
            procs.append(subprocess.Popen([sys.executable, "-u", path], env=env, stdout=fh, stderr=subprocess.STDOUT)); logs.append(lp); fhs.append(fh)
            time.sleep(15 if (torch.cuda.is_available() and not SMOKE) else 1)    # stagger start-up (weight cache / IO)
        codes = _monitor(procs, logs)
    finally:
        for p in procs:
            if p.poll() is None: p.terminate()
        for fh in fhs: fh.close()
    print(f"[PARALLEL] workers exited with codes {codes}")
    if any(c != 0 for c in codes): print("[PARALLEL] WARNING: a worker crashed; see worker*.log. Completed runs are kept; re-run the cell to resume.")


def warm_pretrained_cache():
    """Download the timm weights once, up front (fails early and clearly without Internet)."""
    if not CFG["pretrained"]: return
    for b in sorted({vcfg(v, "backbone") for v in CFG["variants"]}):
        try: timm.create_model(b, pretrained=True, num_classes=0, global_pool="")
        except Exception as e: raise RuntimeError(f"Cannot load pretrained {b}: {e}. Turn Internet ON (or attach the weights).")
        print(f"[WEIGHTS] {b} pretrained weights available")


def main():
    if os.environ.get("WAVESSM_WORKER") is not None:
        worker_main(); return
    if os.environ.get("R2_EVAL_ONLY") == "1":                 # CPU re-analysis of finished runs (R2-6): no training, no GPU
        r2_restore_previous_outputs(); runs = load_runs(); aggregate(runs)
        r2_run_all(CFG["out_dir"], CFG["primary_variant"], CFG["seeds"][0]); return
    if SMOKE: make_smoke_data()
    r2_restore_previous_outputs()
    dump_reproducibility(); test_haar_vs_pywt()
    print(f"[ENV] {torch.cuda.device_count() if torch.cuda.is_available() else 0} GPU(s): " +
          ", ".join(torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())) if torch.cuda.is_available() else "[ENV] no GPU")
    manifest = build_data()
    dump_cv_manifest(manifest)
    warm_pretrained_cache()
    run_jobs(manifest)
    print(f"\n[TIME] training phase finished after {(time.time() - T0) / 3600:.2f} h")
    runs = load_runs(); aggregate(runs)
    if CFG["run_cv"]: cv_aggregate()
    _r2_safe("r2_run_all", r2_run_all, CFG["out_dir"], CFG["primary_variant"], CFG["seeds"][0])      # all reviewer-2 tables/figures from the predictions
    v, s = CFG["primary_variant"], CFG["seeds"][0]
    if CFG["run_analysis"] and os.path.exists(os.path.join(CFG["out_dir"], f"ckpt_{v}_s{s}.pt")):
        model = load_primary(v, s)
        te, tr = manifest[manifest.role == "test"].reset_index(drop=True), manifest[manifest.role == "train"]
        print("[ANALYSIS] attention entropy:", attention_entropy(model, make_loader(te, False)))
        print("[ANALYSIS] complexity/speed:", json.dumps(complexity_and_speed(model), indent=1)[:600])
        occlusion_and_shapley(model, tr, te); gradcam_all_branches(model, te); embeddings(model, te); domain_shift_table(manifest)
        pdf = pd.read_csv(os.path.join(CFG["out_dir"], f"pred_{v}_s{s}_test.csv"))
        _r2_safe("gradcam failures", r2_gradcam_failures, model, pdf, CFG["r2_failure_cases"])
        _r2_safe("gradcam statistics", r2_gradcam_quant, model, te, CFG["r2_gradcam_per_class"])
        if CFG["r2_perturb"]: _r2_safe("perturbations", r2_perturbation_eval, model, te)
        hist = pd.read_csv(os.path.join(CFG["out_dir"], f"history_{v}_s{s}.csv"))
        fig, ax = plt.subplots(1, 3, figsize=(15, 4))
        ax[0].plot(hist.epoch, hist.train_loss, label="train"); ax[0].plot(hist.epoch, hist.val_loss, label="val (held-out)"); ax[0].legend(); ax[0].set_title("Loss")
        ax[1].plot(hist.epoch, 100 * hist.train_acc, label="train"); ax[1].plot(hist.epoch, 100 * hist.val_acc, label="val raw"); ax[1].plot(hist.epoch, 100 * hist.ema_val_acc, label="val EMA"); ax[1].legend(); ax[1].set_title("Accuracy (%)")
        ax[2].plot(np.load(os.path.join(CFG["out_dir"], f"lr_steps_{v}_s{s}.npy"))); ax[2].set_title("LR trajectory (per optimiser step)")
        fig.tight_layout(); fig.savefig(os.path.join(CFG["out_dir"], "fig_training_curves_lr.png"), dpi=300); plt.close(fig)
    elif CFG["run_analysis"]:
        print(f"[ANALYSIS] skipped: checkpoint of the primary run {v}_s{s} does not exist")
    print(f"\n[DONE] outputs in {CFG['out_dir']}   total time {(time.time() - T0) / 3600:.2f} h")


if __name__ == "__main__":
    main()

[CONFIG] RUN_MODE=paper  seeds=[42, 43, 44, 45, 46]  epochs=50  variants=['full', 'softmax_head', 'wavelet_only', 'ssm_only', 'cnn_only', 'wt_ssm_concat', 'ssm_cnn_concat', 'wt_cnn_concat', 'concat_mlp', 'mha_fusion', 'gated_fusion', 'concat_mlp_wide', 'mha_fusion_wide', 'cnn_b4', 'no_wavelet', 'no_ssm', 'full_clamp100']  cv=True
[TEST] Haar DWT vs PyWavelets (LL, LH, HL, HH): [True, True, True, True]
[ENV] 2 GPU(s): Tesla T4, Tesla T4
[DATA] BRISC train=5000  test=1000  class counts test=[254, 306, 140, 300]
[DUP] near-duplicate pairs (cos>=0.985): train-train=153  test-test=26  train-test=176
[DUP] train images sharing a cluster with a test image: 159 (dropped)
[R2] planes in the test split: {'ax': 398, 'co': 305, 'sa': 297}   max cosine similarity test->train: median 0.942
[EXT] 5600 images  per-class counts (VERIFY against the paper's 5,600): [1400, 1400, 1400, 1400]
[EXT] images near-duplicating a BRISC image: 3863 (dropped)
                      glioma  meningioma  notumor  pitui

model.safetensors: reconstructing file:   0%|          |  0.00B / 49.3MB            

model.safetensors: downloading bytes:           |  0.00B            

[WEIGHTS] efficientnet_b3 pretrained weights available


model.safetensors: reconstructing file:   0%|          |  0.00B / 77.9MB            

model.safetensors: downloading bytes:           |  0.00B            

[WEIGHTS] efficientnet_b4 pretrained weights available
[PARALLEL] launching 2 worker processes (one per GPU); jobs in priority order: ['main_full_s42', 'main_softmax_head_s42', 'main_wavelet_only_s42', 'main_ssm_only_s42', 'main_cnn_only_s42', 'main_wt_ssm_concat_s42', 'main_ssm_cnn_concat_s42', 'main_wt_cnn_concat_s42', 'main_concat_mlp_s42', 'main_mha_fusion_s42', 'main_gated_fusion_s42', 'main_concat_mlp_wide_s42', 'main_mha_fusion_wide_s42', 'main_cnn_b4_s42', 'main_no_wavelet_s42', 'main_no_ssm_s42', 'main_full_clamp100_s42', 'main_full_s43', 'main_softmax_head_s43', 'main_wavelet_only_s43', 'main_ssm_only_s43', 'main_cnn_only_s43', 'main_wt_ssm_concat_s43', 'main_ssm_cnn_concat_s43', 'main_wt_cnn_concat_s43', 'main_concat_mlp_s43', 'main_mha_fusion_s43', 'main_gated_fusion_s43', 'main_concat_mlp_wide_s43', 'main_mha_fusion_wide_s43', 'main_cnn_b4_s43', 'main_no_wavelet_s43', 'main_no_ssm_s43', 'main_full_clamp100_s43', 'main_full_s44', 'main_softmax_head_s44', 'main_wavelet_only_

I0000 00:00:1793805587.607145      23 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


                           mean                 std             bg_frac  \
                           mean       std      mean       std      mean   
set        class                                                          
brisc_test glioma      0.135132  0.032902  0.151415  0.020880  0.510736   
           meningioma  0.205344  0.084575  0.198367  0.045119  0.419947   
           notumor     0.187305  0.047557  0.199523  0.039881  0.465074   
           pituitary   0.186392  0.031501  0.158756  0.019309  0.353323   
external   glioma      0.129284  0.031360  0.144468  0.020206  0.510803   
           meningioma  0.168761  0.041908  0.171112  0.018581  0.438109   
           notumor     0.250381  0.089991  0.232987  0.060913  0.348383   
           pituitary   0.268199  0.045680  0.178014  0.018392  0.164100   

                                  fg_mean                 p95            
                            std      mean       std      mean       std  
set        class          